# Concentração das exportações brasileiras (HHI), 2000-2025

Monografia de Lucas Massayuki Libanio Iwakawa, Ciências Econômicas, PUC Minas, 2026.

Tratamento dos microdados de exportação do Comex Stat (MDIC), NCM a oito dígitos, de 2000 a 2025, e cálculo dos índices usados na monografia.

1 a 8. Leitura dos arquivos, tabelas auxiliares, exclusões e conferência com os totais do MDIC
9. HHI e razões de concentração
10 e 11. Preços constantes e sensibilidade ao ano-base
12 e 13. Extratos e planilha de resultados
14. Série corrente no universo do vetor, teste de sinal, testes de quebra, UNCTAD e FMI
15. Registro do teste do vetor de preços anterior (desligado por padrão)

In [1]:
import os
import glob
import re
import unicodedata
import numpy as np
import pandas as pd

# Configuração
PASTA_DADOS = r"G:\My Drive\Puc\8 semestre\Monografia\Segundo Tema\Bds"

# Saídas em pasta separada dos dados de entrada
PASTA_SAIDA = os.path.join(PASTA_DADOS, "..", "dados_tratados")
PASTA_SAIDA = os.path.normpath(PASTA_SAIDA)
os.makedirs(PASTA_SAIDA, exist_ok=True)

PASTA_EXTRATOS = os.path.join(PASTA_SAIDA, "extratos")
os.makedirs(PASTA_EXTRATOS, exist_ok=True)

ARQ_AUXILIAR = os.path.join(PASTA_DADOS, "TABELAS_AUXILIARES.xlsx")

SEP_CSV = ";"
ENC_CSV = "latin-1"

ANO_INI, ANO_FIM = 2000, 2025
ANO_BASE_PRECOS = 2017             # ano-base principal da série a preços constantes

# Escolhas metodológicas
NIVEL_PRINCIPAL = "SH4"            # posição do Sistema Harmonizado (4 dígitos)
NIVEIS_ROBUSTEZ = ["CAP2", "SH6", "CO_NCM"]

# Vetor de preços do ano-base
NIVEL_VETOR_PRECOS = "SH4"         # posição SH-4: não muda com as revisões do SH
MASSA_MIN_KG = 1_000               # massa mínima por linha no ano-base
# Posições em que o quilograma não mede quantidade, fora do vetor de preços
EXCLUIR_CAPITULOS = ["71", "89"]    # pedras e metais preciosos; embarcações
EXCLUIR_POSICOES = ["8802", "8411"] # aviões e demais aeronaves; turborreatores e turbinas

# Anos-base alternativos para o teste de sensibilidade do efeito preço
ANOS_BASE_TESTE = [2005, 2010, 2017]

# Categorias de NO_FAT_AGREG que não são produto, e sim operação especial
CATEGORIAS_OPERACAO_ESPECIAL = ["CONSUMO DE BORDO", "TRANSACOES ESPECIAIS"]

# Limiar de alerta: participação do maior item em qualquer série
LIMIAR_ALERTA_SMAX = 0.25

# Seção 15: teste do vetor de preços anterior (demora alguns minutos)
RODAR_REGISTRO_VETOR = False

print("Entradas :", PASTA_DADOS)
print("Saídas   :", PASTA_SAIDA)
print("Anos     :", ANO_INI, "a", ANO_FIM)
print("Nível principal :", NIVEL_PRINCIPAL, "| robustez:", ", ".join(NIVEIS_ROBUSTEZ))
print("Anos-base testados no efeito preço:", ANOS_BASE_TESTE, f"(principal: {ANO_BASE_PRECOS})")
print(f"Vetor de preços: {NIVEL_VETOR_PRECOS} | exclusões: capítulos {EXCLUIR_CAPITULOS}, posições {EXCLUIR_POSICOES}")

Entradas : G:\My Drive\Puc\8 semestre\Monografia\Segundo Tema\Bds
Saídas   : G:\My Drive\Puc\8 semestre\Monografia\Segundo Tema\dados_tratados
Anos     : 2000 a 2025
Nível principal : SH4 | robustez: CAP2, SH6, CO_NCM
Anos-base testados no efeito preço: [2005, 2010, 2017] (principal: 2017)
Vetor de preços: SH4 | exclusões: capítulos ['71', '89'], posições ['8802', '8411']


## 1. Arquivos anuais

O padrão de nome aceita apenas `EXP_` seguido de quatro dígitos, para não capturar arquivos consolidados gravados na mesma pasta (como `EXP_TODOS_JUNTOS.csv`).

In [2]:
padrao = os.path.join(PASTA_DADOS, "EXP_[0-9][0-9][0-9][0-9].csv")
arquivos = sorted(glob.glob(padrao))

anos_encontrados = sorted(int(re.search(r"EXP_(\d{4})\.csv$", os.path.basename(a)).group(1))
                          for a in arquivos)
anos_esperados = list(range(ANO_INI, ANO_FIM + 1))
faltantes = sorted(set(anos_esperados) - set(anos_encontrados))
excedentes = sorted(set(anos_encontrados) - set(anos_esperados))

print(f"Arquivos encontrados: {len(arquivos)}")
print(f"Anos: {anos_encontrados[0]}-{anos_encontrados[-1]}")
if faltantes:
    print(f"Anos faltantes: {faltantes}")
if excedentes:
    print(f"Anos fora do recorte: {excedentes}")

assert not faltantes, f"Baixar os arquivos dos anos: {faltantes}"
assert all("TODOS" not in a.upper() for a in arquivos), "Arquivo consolidado capturado pelo glob"

Arquivos encontrados: 26
Anos: 2000-2025


## 2. Leitura com agregação por arquivo

`CO_NCM` é lido como texto e completado com zeros à esquerda até oito dígitos. Lido como inteiro, o código perde o zero inicial (`08072000` vira `8072000`) e o recorte por capítulo passa a usar os dígitos errados, o que afeta os capítulos 01 a 09 (café, soja, carnes e frutas).

Cada arquivo anual é agregado por `CO_ANO`, `CO_NCM` e `CO_PAIS` logo na leitura.

In [3]:
DTYPES = {
    "CO_ANO": "int16", "CO_MES": "int8", "CO_NCM": "str", "CO_UNID": "int16",
    "CO_PAIS": "int16", "SG_UF_NCM": "str", "CO_VIA": "int8", "CO_URF": "int32",
    "QT_ESTAT": "int64", "KG_LIQUIDO": "int64", "VL_FOB": "int64",
}
CHAVES = ["CO_ANO", "CO_NCM", "CO_PAIS"]
MEDIDAS = ["VL_FOB", "KG_LIQUIDO"]

partes, auditoria = [], []

for caminho in arquivos:
    nome = os.path.basename(caminho)
    df = pd.read_csv(caminho, sep=SEP_CSV, encoding=ENC_CSV, dtype=DTYPES)

    # Normalização da chave: 8 dígitos, zeros à esquerda preservados
    df["CO_NCM"] = df["CO_NCM"].str.strip().str.zfill(8)
    assert df["CO_NCM"].str.len().eq(8).all(), f"{nome}: NCM fora do padrão de 8 dígitos"

    auditoria.append({
        "arquivo": nome,
        "linhas_brutas": len(df),
        "fob_bruto": int(df["VL_FOB"].sum()),
        "ncm_distintas": df["CO_NCM"].nunique(),
        "paises_distintos": df["CO_PAIS"].nunique(),
    })

    partes.append(df.groupby(CHAVES, as_index=False)[MEDIDAS].sum())
    print(f"{nome}: {len(df):>10,} linhas -> {len(partes[-1]):>7,} agregadas")

    del df

auditoria = pd.DataFrame(auditoria)
base = pd.concat(partes, ignore_index=True)
del partes

print(f"\nBase agregada: {len(base):,} linhas")
print(f"Linhas nos microdados: {auditoria['linhas_brutas'].sum():,}")

EXP_2000.csv:    608,920 linhas ->  77,992 agregadas
EXP_2001.csv:    642,737 linhas ->  82,169 agregadas
EXP_2002.csv:    652,868 linhas ->  91,275 agregadas
EXP_2003.csv:    756,433 linhas -> 100,763 agregadas
EXP_2004.csv:    872,417 linhas -> 109,401 agregadas
EXP_2005.csv:  1,082,314 linhas -> 128,021 agregadas
EXP_2006.csv:  1,060,518 linhas -> 122,680 agregadas
EXP_2007.csv:  1,107,634 linhas -> 128,140 agregadas
EXP_2008.csv:  1,085,591 linhas -> 123,139 agregadas
EXP_2009.csv:  1,006,104 linhas -> 119,448 agregadas
EXP_2010.csv:  1,044,917 linhas -> 118,953 agregadas
EXP_2011.csv:  1,071,261 linhas -> 119,466 agregadas
EXP_2012.csv:  1,063,549 linhas -> 118,330 agregadas
EXP_2013.csv:  1,074,709 linhas -> 117,511 agregadas
EXP_2014.csv:  1,057,794 linhas -> 116,966 agregadas
EXP_2015.csv:  1,048,388 linhas -> 117,039 agregadas
EXP_2016.csv:  1,092,692 linhas -> 120,960 agregadas
EXP_2017.csv:  1,096,623 linhas -> 119,427 agregadas
EXP_2018.csv:  1,128,981 linhas -> 136,725 agr

### Conferência da agregação

A soma do FOB dos arquivos individuais deve ser igual à soma da base agregada.

In [4]:
fob_arquivos = auditoria["fob_bruto"].sum()
fob_base = int(base["VL_FOB"].sum())

print(f"FOB somado dos arquivos : {fob_arquivos:>20,}")
print(f"FOB da base agregada    : {fob_base:>20,}")
print(f"Diferença               : {fob_arquivos - fob_base:>20,}")

assert fob_arquivos == fob_base, "Perda ou duplicação de valor na agregação"
assert not base.duplicated(subset=CHAVES).any(), "Chave ano×NCM×país duplicada"

display(auditoria)

FOB somado dos arquivos :    5,135,607,346,029
FOB da base agregada    :    5,135,607,346,029
Diferença               :                    0


,arquivo,linhas_brutas,fob_bruto,ncm_distintas,paises_distintos
0,EXP_2000.csv,608920,54993159648,6717,211
1,EXP_2001.csv,642737,58032294243,6832,213
2,EXP_2002.csv,652868,60147158103,7295,213
3,EXP_2003.csv,756433,72776746690,7101,221
4,EXP_2004.csv,872417,95121672369,7185,221
5,EXP_2005.csv,1082314,118597835407,7553,231
6,EXP_2006.csv,1060518,137581151209,7344,236
7,EXP_2007.csv,1107634,159816383833,7767,231
8,EXP_2008.csv,1085591,195764624177,7404,227
9,EXP_2009.csv,1006104,151791674186,7310,228


## 3. Tabelas auxiliares

As chaves `CO_NCM` das planilhas auxiliares chegam do Excel como número e recebem a mesma normalização para oito dígitos. Antes dos cruzamentos, verifica-se se a chave é única em cada tabela, já que uma chave duplicada multiplicaria linhas no merge.

In [5]:
def norm_ncm(serie):
    return serie.astype("str").str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(8)

aba_ncm  = pd.read_excel(ARQ_AUXILIAR, sheet_name="NCM")
aba_cgce = pd.read_excel(ARQ_AUXILIAR, sheet_name="CGEs")
aba_ppe  = pd.read_excel(ARQ_AUXILIAR, sheet_name="Prod exp")
aba_pais = pd.read_excel(ARQ_AUXILIAR, sheet_name="País")

for nome, aba in [("NCM", aba_ncm), ("CGEs", aba_cgce), ("Prod exp", aba_ppe)]:
    aba["CO_NCM"] = norm_ncm(aba["CO_NCM"])
    dup = aba["CO_NCM"].duplicated().sum()
    print(f"{nome:<10} {len(aba):>7,} linhas | chaves duplicadas: {dup}")
    assert dup == 0, f"Aba {nome}: chave CO_NCM duplicada"

# A aba de países pode trazer variações de nomenclatura nas colunas
col_pais_nome = next(c for c in ["NO_PAIS", "NO_PAIS_POR", "NOME_PAIS"] if c in aba_pais.columns)
aba_pais = aba_pais.rename(columns={col_pais_nome: "NO_PAIS"})
aba_pais["CO_PAIS"] = pd.to_numeric(aba_pais["CO_PAIS"], errors="coerce").astype("Int64")
assert aba_pais["CO_PAIS"].duplicated().sum() == 0, "Aba País: CO_PAIS duplicado"
print(f"{'País':<10} {len(aba_pais):>7,} linhas | coluna de nome: {col_pais_nome}")
print("\nColunas disponíveis em País:", list(aba_pais.columns))

NCM         13,705 linhas | chaves duplicadas: 0
CGEs        13,726 linhas | chaves duplicadas: 0
Prod exp    13,423 linhas | chaves duplicadas: 0
País           281 linhas | coluna de nome: NO_PAIS

Colunas disponíveis em País: ['CO_PAIS', 'CO_PAIS_ISON3', 'CO_PAIS_ISOA3', 'NO_PAIS', 'NO_PAIS_ING', 'NO_PAIS_ESP']


## 4. Cruzamentos

Depois de cada merge, confere-se se o número de linhas se manteve e calcula-se a parcela do FOB que ficou sem classificação (NCMs descontinuadas, ausentes da tabela auxiliar vigente).

In [6]:
n_ref = len(base)
fob_total = base["VL_FOB"].sum()

def merge_verificado(esq, dir_, chave, rotulo):
    out = esq.merge(dir_, on=chave, how="left")
    assert len(out) == n_ref, (
        f"{rotulo}: {len(out):,} linhas após o merge, {n_ref:,} esperadas"
    )
    nova = [c for c in dir_.columns if c != chave][0]
    fob_nulo = out.loc[out[nova].isna(), "VL_FOB"].sum()
    print(f"{rotulo:<14} ok | FOB sem classificação: {fob_nulo / fob_total:.4%}")
    return out

base = merge_verificado(base, aba_ncm[["CO_NCM", "NO_FAT_AGREG"]], "CO_NCM", "Fator agregado")
base = merge_verificado(base, aba_cgce[["CO_NCM", "NO_CGCE_N3"]], "CO_NCM", "CGCE N3")
base = merge_verificado(base, aba_ppe[["CO_NCM", "NO_PPE"]], "CO_NCM", "PPE")
base = merge_verificado(base, aba_pais[["CO_PAIS", "NO_PAIS"]], "CO_PAIS", "País")

assert base["VL_FOB"].sum() == fob_total, "FOB alterado por algum merge"
print(f"\nBase final: {len(base):,} linhas x {base.shape[1]} colunas")

Fator agregado ok | FOB sem classificação: 0.0050%
CGCE N3        ok | FOB sem classificação: 0.0001%
PPE            ok | FOB sem classificação: 0.2148%
País           ok | FOB sem classificação: 0.0000%

Base final: 3,321,548 linhas x 9 colunas


## 5. Níveis de agregação

Capítulo (2 dígitos), posição (SH-4) e subposição (SH-6) são obtidos por corte do código NCM. O nível principal é o SH-4 (seção 3.2 da monografia); os demais são usados como teste de robustez.

In [7]:
base["CAP2"] = base["CO_NCM"].str[:2]
base["SH4"]  = base["CO_NCM"].str[:4]
base["SH6"]  = base["CO_NCM"].str[:6]

for nivel in [NIVEL_PRINCIPAL] + NIVEIS_ROBUSTEZ:
    ativos = base[base["VL_FOB"] > 0].groupby("CO_ANO")[nivel].nunique()
    marca = " (principal)" if nivel == NIVEL_PRINCIPAL else ""
    print(f"{nivel:<7} N ativos: {ativos.min():>5} (min) {ativos.max():>5} (max) "
          f"| variação: {ativos.max() / ativos.min():.2f}x{marca}")

SH4     N ativos:  1179 (min)  1214 (max) | variação: 1.03x (principal)
CAP2    N ativos:    96 (min)    97 (max) | variação: 1.01x
SH6     N ativos:  4340 (min)  4963 (max) | variação: 1.14x
CO_NCM  N ativos:  6713 (min)  8189 (max) | variação: 1.22x


## 6. Operações especiais

O campo `NO_FAT_AGREG` tem cinco valores, mas só três são categorias de produto (básicos, semimanufaturados e manufaturados). Consumo de bordo e transações especiais são operações especiais e deixam de ter registros a partir de 2020:

| Categoria | 2018 | 2019 | 2020 |
|---|---|---|---|
| Consumo de bordo | 2,20 bi | 8,4 mi | 0 |
| Transações especiais | 24,0 mi | 399 | 0 |

Mantidas na base, elas fariam o N do fator agregado cair de 5 para 3 em 2020 e criariam uma descontinuidade nas séries de produto. Por isso são excluídas de todas as séries. A base completa (`base`) fica disponível para conferência, e a análise usa `base_analise`.

NCMs sem classificação são mantidas: são produtos ausentes da tabela auxiliar e somam menos de 0,01% do valor.

In [8]:
base["OPERACAO_ESPECIAL"] = base["NO_FAT_AGREG"].isin(CATEGORIAS_OPERACAO_ESPECIAL)

resumo_especial = (base.loc[base["OPERACAO_ESPECIAL"]]
                       .groupby(["NO_FAT_AGREG", "CO_ANO"])["VL_FOB"].sum()
                       .unstack("NO_FAT_AGREG").fillna(0))

fob_total_base = base["VL_FOB"].sum()
pct_especial = base.loc[base["OPERACAO_ESPECIAL"], "VL_FOB"].sum() / fob_total_base

print("Valor por categoria de operação especial (US$ bi):")
display((resumo_especial / 1e9).round(3))
print(f"\nExcluído do conjunto de análise: {pct_especial:.4%} do FOB do período")

base_analise = base.loc[~base["OPERACAO_ESPECIAL"]].copy()

restantes = sorted(base_analise["NO_FAT_AGREG"].dropna().unique())
print(f"\nCategorias de fator agregado restantes ({len(restantes)}): {restantes}")

n_por_ano = base_analise.dropna(subset=["NO_FAT_AGREG"]).groupby("CO_ANO")["NO_FAT_AGREG"].nunique()
assert n_por_ano.nunique() == 1, f"N de fator agregado ainda varia: {n_por_ano.to_dict()}"
print(f"N constante em {n_por_ano.iloc[0]} categorias nos {len(n_por_ano)} anos")

print(f"\nbase          : {len(base):,} linhas (completa, auditoria)")
print(f"base_analise  : {len(base_analise):,} linhas (produto e classificações)")

Valor por categoria de operação especial (US$ bi):


NO_FAT_AGREG,CONSUMO DE BORDO,TRANSACOES ESPECIAIS
CO_ANO,,
2000,1.034,0.039
2001,1.130,0.056
2002,0.987,0.056
2003,1.173,0.056
2004,1.402,0.069
2005,2.101,0.078
2006,2.526,0.088
2007,2.898,0.097
2008,4.725,0.095



Excluído do conjunto de análise: 1.0726% do FOB do período

Categorias de fator agregado restantes (3): ['PRODUTOS BASICOS', 'PRODUTOS MANUFATURADOS', 'PRODUTOS SEMIMANUFATURADOS']
N constante em 3 categorias nos 26 anos

base          : 3,321,548 linhas (completa, auditoria)
base_analise  : 3,311,730 linhas (produto e classificações)


## 7. Códigos residuais de país de destino

Alguns códigos de `CO_PAIS` não correspondem a mercados de destino: provisão de navios e aeronaves, reexportação, destino a designar e não declarado. Eles são excluídos da dimensão de destino (seção 3.4 da monografia), com registro da parcela do valor afetada.

A identificação é feita por padrão de texto em `NO_PAIS`, porque os códigos mudam entre versões da tabela auxiliar. A lista impressa abaixo deve ser conferida sempre que a tabela for atualizada.

In [9]:
PADROES_PAIS_RESIDUAL = [
    "PROVISAO",              # provisão de navios e aeronaves
    "CONSUMO DE BORDO",
    "REEXPORTAC",
    "NAO DECLARAD",
    "A DESIGNAR",
    "NAO ESPECIFICAD",
    "NAO DETERMINAD",
]


def sem_acento(texto):
    normalizado = unicodedata.normalize("NFKD", str(texto))
    return "".join(c for c in normalizado if not unicodedata.combining(c)).upper()


base["PAIS_NORM"] = base["NO_PAIS"].map(sem_acento)
base["PAIS_RESIDUAL"] = base["PAIS_NORM"].str.contains(
    "|".join(PADROES_PAIS_RESIDUAL), na=False, regex=True
)
base_analise["PAIS_RESIDUAL"] = base_analise.index.map(base["PAIS_RESIDUAL"])

resumo_residual = (base.loc[base["PAIS_RESIDUAL"]]
                       .groupby("NO_PAIS")["VL_FOB"].sum()
                       .sort_values(ascending=False))

print("Destinos classificados como residuais:\n")
for nome, fob in resumo_residual.items():
    print(f"  {nome:<45} US$ {fob/1e9:>8.2f} bi  ({fob/fob_total_base:.4%})")

pct_excluido = resumo_residual.sum() / fob_total_base
print(f"\nTotal excluído da dimensão de destino: {pct_excluido:.4%} do FOB")

# Destinos de menor valor não classificados como residuais, para conferência
print("\n15 destinos de menor valor não classificados como residuais:")
menores = (base.loc[~base["PAIS_RESIDUAL"]]
               .groupby("NO_PAIS")["VL_FOB"].sum().sort_values().head(15))
for nome, fob in menores.items():
    print(f"  {nome:<45} US$ {fob:>15,.0f}")

# A base de destino parte da base de análise, já sem operações especiais
base_destino = base_analise.loc[~base_analise["PAIS_RESIDUAL"]].copy()
print(f"\nbase_analise  : {len(base_analise):,} linhas (produto)")
print(f"base_destino  : {len(base_destino):,} linhas (destino, sem residuais)")

Destinos classificados como residuais:

  Provisão de Navios e Aeronaves                US$    49.60 bi  (0.9659%)
  A Designar                                    US$     0.00 bi  (0.0001%)

Total excluído da dimensão de destino: 0.9659% do FOB

15 destinos de menor valor não classificados como residuais:
  Bancos Centrais                               US$             196
  Wake, Ilha                                    US$           5,153
  Svalbard e Jan Mayen                          US$           6,093
  Canal, Ilhas do (Jersey)                      US$          11,108
  Midway, Ilhas                                 US$          41,073
  Guernsey                                      US$          95,333
  Aland, Ilhas                                  US$         114,681
  Lebuan, Ilhas                                 US$         129,130
  Geórgia do Sul e Sandwich do Sul, Ilhas       US$         135,180
  Jersey                                        US$         179,550
  Johnston, I

### Gravação da base tratada

In [10]:
caminho_parquet = os.path.join(PASTA_SAIDA, "base_agregada.parquet")
try:
    base.to_parquet(caminho_parquet, index=False)
    print(f"Base salva em: {caminho_parquet}")
except ImportError:
    caminho_parquet = os.path.join(PASTA_SAIDA, "base_agregada.csv.gz")
    base.to_csv(caminho_parquet, index=False, sep=SEP_CSV, compression="gzip")
    print(f"pyarrow não instalado; base salva em CSV comprimido: {caminho_parquet}")

pyarrow não instalado; base salva em CSV comprimido: G:\My Drive\Puc\8 semestre\Monografia\Segundo Tema\dados_tratados\base_agregada.csv.gz


## 8. Conferência com os totais do MDIC

As verificações anteriores são internas. Para comparar a base lida com a base publicada, usa-se o arquivo `EXP_TOTAIS_CONFERENCIA.csv`, que o MDIC divulga junto com os microdados, com o número de linhas e o valor FOB de cada ano.

Fonte: <https://www.gov.br/mdic/pt-br/assuntos/comercio-exterior/estatisticas/base-de-dados-bruta>

A conferência usa `base`, antes das exclusões das seções 6 e 7, porque é ela que deve reproduzir o total publicado. Se houver divergência, as causas mais prováveis são arquivos baixados em datas diferentes (o MDIC revisa os dados) ou o tratamento das exportações fictas do REPETRO, que afeta sobretudo 2018 e 2019.

In [11]:
import urllib.request, datetime

URL_CONFERENCIA = ("https://balanca.mdic.gov.br/balanca/bd/comexstat-bd/ncm/"
                   "EXP_TOTAIS_CONFERENCIA.csv")
ARQ_CONFERENCIA = os.path.join(PASTA_DADOS, "EXP_TOTAIS_CONFERENCIA.csv")

if not os.path.exists(ARQ_CONFERENCIA):
    try:
        urllib.request.urlretrieve(URL_CONFERENCIA, ARQ_CONFERENCIA)
    except Exception as e:
        raise SystemExit(f"Download falhou ({type(e).__name__}). Baixe o arquivo em "
                         f"{URL_CONFERENCIA} e salve em {ARQ_CONFERENCIA}.")

DATA_CONFERENCIA = datetime.datetime.fromtimestamp(
    os.path.getmtime(ARQ_CONFERENCIA)).strftime("%d/%m/%Y")
print(f"Data do arquivo de conferência: {DATA_CONFERENCIA}")

conf_bruta = pd.read_csv(ARQ_CONFERENCIA, sep=SEP_CSV, encoding=ENC_CSV)
conf_bruta.columns = [c.strip().upper() for c in conf_bruta.columns]

print("\nColunas:", list(conf_bruta.columns))
display(conf_bruta.head())

Data do arquivo de conferência: 05/10/2026

Colunas: ['ARQUIVO', 'CO_ANO', 'QT_ESTAT', 'KG_LIQUIDO', 'VL_FOB', 'NUMERO_LINHAS']


,ARQUIVO,CO_ANO,QT_ESTAT,KG_LIQUIDO,VL_FOB,NUMERO_LINHAS
0,EXP_1997.csv,1997,12627973856,208817826940,52947495532,529815
1,EXP_1998.csv,1998,46292246885,230153042546,51076603549,531069
2,EXP_1999.csv,1999,181308756553,228543547383,47945909310,549466
3,EXP_2000.csv,2000,71571312327,244537401277,54993159648,608920
4,EXP_2001.csv,2001,827374586980,272545750885,58032294243,642737


In [12]:
# Os nomes das colunas do arquivo de conferência mudam entre atualizações do MDIC;
# por isso as colunas são localizadas por palavra-chave.

def _achar(colunas, *chaves):
    for c in colunas:
        if any(k in c for k in chaves):
            return c
    return None


col_ano   = _achar(conf_bruta.columns, "ANO")
col_fob   = _achar(conf_bruta.columns, "FOB", "VALOR")
col_linha = _achar(conf_bruta.columns, "LINHA", "REGISTRO", "QTD", "QUANT")

assert col_ano and col_fob, (
    f"Colunas de ano e FOB não identificadas em {list(conf_bruta.columns)}"
)

renomear = {col_ano: "CO_ANO", col_fob: "fob_oficial"}
if col_linha:
    renomear[col_linha] = "linhas_oficiais"

conf = conf_bruta.rename(columns=renomear)
conf["CO_ANO"] = pd.to_numeric(conf["CO_ANO"], errors="coerce").astype("Int64")
conf["fob_oficial"] = pd.to_numeric(
    conf["fob_oficial"].astype(str).str.replace(".", "", regex=False)
                       .str.replace(",", ".", regex=False),
    errors="coerce")

conf = conf.loc[conf["CO_ANO"].between(ANO_INI, ANO_FIM)].copy()

print(f"Coluna de ano   : {col_ano}")
print(f"Coluna de FOB   : {col_fob}")
print(f"Coluna de linhas: {col_linha or '(ausente; conferência só do FOB)'}")
print(f"Anos de referência no recorte: {conf['CO_ANO'].min()}-{conf['CO_ANO'].max()} "
      f"({len(conf)} de {ANO_FIM - ANO_INI + 1})")


Coluna de ano   : CO_ANO
Coluna de FOB   : VL_FOB
Coluna de linhas: NUMERO_LINHAS
Anos de referência no recorte: 2000-2025 (26 de 26)


In [13]:
TOLERANCIA_PCT = 0.01      # em %

por_ano = (base.groupby("CO_ANO")["VL_FOB"].sum()
               .rename("fob_calculado").reset_index())
por_ano["CO_ANO"] = por_ano["CO_ANO"].astype("Int64")

cols_conf = ["CO_ANO", "fob_oficial"] + (["linhas_oficiais"] if col_linha else [])
val = por_ano.merge(conf[cols_conf], on="CO_ANO", how="left")

val["dif_abs"] = val["fob_calculado"] - val["fob_oficial"]
val["dif_pct"] = val["dif_abs"] / val["fob_oficial"] * 100

display(val[["CO_ANO", "fob_calculado", "fob_oficial", "dif_abs", "dif_pct"]]
        .style.format({"fob_calculado": "{:,.0f}", "fob_oficial": "{:,.0f}",
                       "dif_abs": "{:,.0f}", "dif_pct": "{:+.6f}%"}))

sem_ref = val.loc[val["fob_oficial"].isna(), "CO_ANO"].tolist()
if sem_ref:
    print(f"Anos sem total oficial: {sem_ref}")

fob_calc_tot = val.loc[val["fob_oficial"].notna(), "fob_calculado"].sum()
fob_ofi_tot  = val["fob_oficial"].sum()
DIVERGENCIA_AGREGADA = (fob_calc_tot / fob_ofi_tot - 1) * 100

print(f"\nFOB calculado (anos com referência): {fob_calc_tot:,.0f}")
print(f"FOB oficial   (anos com referência): {fob_ofi_tot:,.0f}")
print(f"Divergência agregada: {DIVERGENCIA_AGREGADA:+.6f}%")

fora = val.loc[val["dif_pct"].abs() > TOLERANCIA_PCT]

if len(fora):
    print(f"\nANOS FORA DA TOLERÂNCIA ({TOLERANCIA_PCT}%):")
    display(fora[["CO_ANO", "dif_abs", "dif_pct"]])
    print("\nPossíveis causas: arquivo anual baixado em data diferente da do arquivo de")
    print("conferência, REPETRO (2018 e 2019) ou leitura incompleta do CSV.")
else:
    print("\nTodos os anos dentro da tolerância.")

assert not len(fora), "Divergência acima da tolerância"


,CO_ANO,fob_calculado,fob_oficial,dif_abs,dif_pct
0,2000,"54,993,159,648","54,993,159,648",0,+0.000000%
1,2001,"58,032,294,243","58,032,294,243",0,+0.000000%
2,2002,"60,147,158,103","60,147,158,103",0,+0.000000%
3,2003,"72,776,746,690","72,776,746,690",0,+0.000000%
4,2004,"95,121,672,369","95,121,672,369",0,+0.000000%
5,2005,"118,597,835,407","118,597,835,407",0,+0.000000%
6,2006,"137,581,151,209","137,581,151,209",0,+0.000000%
7,2007,"159,816,383,833","159,816,383,833",0,+0.000000%
8,2008,"195,764,624,177","195,764,624,177",0,+0.000000%
9,2009,"151,791,674,186","151,791,674,186",0,+0.000000%



FOB calculado (anos com referência): 5,135,607,346,029
FOB oficial   (anos com referência): 5,135,607,346,029
Divergência agregada: +0.000000%

Todos os anos dentro da tolerância.


In [14]:
# Contagem de linhas por ano: detecta leitura incompleta mesmo quando as linhas
# perdidas têm valor pequeno.

if col_linha:
    lin = auditoria[["arquivo", "linhas_brutas"]].copy()
    lin["CO_ANO"] = lin["arquivo"].str.extract(r"EXP_(\d{4})").astype(int).astype("Int64")
    lin = lin.merge(conf[["CO_ANO", "linhas_oficiais"]], on="CO_ANO", how="left")
    lin["linhas_oficiais"] = pd.to_numeric(lin["linhas_oficiais"], errors="coerce")
    lin["dif_linhas"] = lin["linhas_brutas"] - lin["linhas_oficiais"]

    display(lin[["CO_ANO", "linhas_brutas", "linhas_oficiais", "dif_linhas"]])

    n_dif = int((lin["dif_linhas"].fillna(0) != 0).sum())
    print(f"\nAnos com contagem divergente: {n_dif}")
    assert n_dif == 0, "Contagem de linhas diferente da oficial"
    print("Contagem de linhas confere em todos os anos.")
else:
    lin = None
    print("Arquivo de conferência sem contagem de linhas; conferência só do FOB.")


,CO_ANO,linhas_brutas,linhas_oficiais,dif_linhas
0,2000,608920,608920,0
1,2001,642737,642737,0
2,2002,652868,652868,0
3,2003,756433,756433,0
4,2004,872417,872417,0
5,2005,1082314,1082314,0
6,2006,1060518,1060518,0
7,2007,1107634,1107634,0
8,2008,1085591,1085591,0
9,2009,1006104,1006104,0



Anos com contagem divergente: 0
Contagem de linhas confere em todos os anos.


In [15]:
# Números citados no texto da monografia

numeros_texto = {
    "registros_microdados":   int(auditoria["linhas_brutas"].sum()),
    "linhas_base_agregada":   int(len(base)),
    "linhas_base_analise":    int(len(base_analise)),
    "linhas_base_destino":    int(len(base_destino)),
    "fob_total_usd":          int(base["VL_FOB"].sum()),
    "fob_total_trilhoes":     round(base["VL_FOB"].sum() / 1e12, 2),
    "recorte":                f"{ANO_INI}-{ANO_FIM}",
    "n_anos":                 ANO_FIM - ANO_INI + 1,
    "fob_ultimo_ano_usd":     int(base.loc[base["CO_ANO"] == ANO_FIM, "VL_FOB"].sum()),
    "pct_operacoes_especiais": f"{pct_especial:.4%}",
    "pct_paises_residuais":    f"{pct_excluido:.4%}",
    "divergencia_oficial":     f"{DIVERGENCIA_AGREGADA:+.6f}%",
    "data_conferencia_mdic":   DATA_CONFERENCIA,
    "vetor_precos":            f"{NIVEL_VETOR_PRECOS}, ano-base {ANO_BASE_PRECOS}",
    "exclusoes_vetor":         f"capítulos {', '.join(EXCLUIR_CAPITULOS)}; posições {', '.join(EXCLUIR_POSICOES)}",
}

for k, v in numeros_texto.items():
    print(f"{k:<26} {v:,}" if isinstance(v, int) else f"{k:<26} {v}")

caminho_numeros = os.path.join(PASTA_SAIDA, "numeros_para_o_texto.csv")
pd.Series(numeros_texto).to_csv(caminho_numeros, header=False)
print(f"\nSalvo em: {caminho_numeros}")


registros_microdados       29,145,544
linhas_base_agregada       3,321,548
linhas_base_analise        3,311,730
linhas_base_destino        3,311,449
fob_total_usd              5,135,607,346,029
fob_total_trilhoes         5.14
recorte                    2000-2025
n_anos                     26
fob_ultimo_ano_usd         348,278,462,500
pct_operacoes_especiais    1.0726%
pct_paises_residuais       0.9659%
divergencia_oficial        +0.000000%
data_conferencia_mdic      05/10/2026
vetor_precos               SH4, ano-base 2017
exclusoes_vetor            capítulos 71, 89; posições 8802, 8411

Salvo em: G:\My Drive\Puc\8 semestre\Monografia\Segundo Tema\dados_tratados\numeros_para_o_texto.csv


## 9. HHI e razões de concentração

$$H = \sum_{i=1}^{N} s_i^2, \qquad s_i = \frac{X_i}{\sum_j X_j}, \qquad H^* = \frac{H - 1/N}{1 - 1/N}$$

$$CR_k = \sum_{i=1}^{k} s_{(i)}, \quad s_{(1)} \ge s_{(2)} \ge \dots$$

`N` é o número de itens com valor exportado positivo no ano (seção 3.4 da monografia). Além do HHI bruto e do normalizado, calculam-se CR4, CR10 e a participação do maior item (`s_max`), usada para identificar séries dominadas por um único item.

A dimensão de destino usa `base_destino`; as demais, `base_analise`.

In [16]:
def calcular_hhi(df, chave, valor="VL_FOB", ano="CO_ANO"):
    """HHI bruto e normalizado, razões de concentração e participação máxima."""
    d = df.loc[df[valor] > 0].groupby([ano, chave], as_index=False)[valor].sum()
    d["s"] = d[valor] / d.groupby(ano)[valor].transform("sum")

    r = (d.groupby(ano)
           .agg(N=("s", "size"),
                H=("s", lambda x: np.sum(x ** 2)),
                CR4=("s", lambda x: x.nlargest(4).sum()),
                CR10=("s", lambda x: x.nlargest(10).sum()),
                s_max=("s", "max"))
           .reset_index())

    # H* é indefinido para N = 1 (pauta monoproduto); por convenção, atribui-se 1
    r["H_norm"] = np.where(r["N"] > 1,
                           (r["H"] - 1 / r["N"]) / (1 - 1 / r["N"]),
                           1.0)
    r["dimensao"] = chave
    return r[[ano, "dimensao", "N", "H", "H_norm", "CR4", "CR10", "s_max"]]


DIMENSOES_PRODUTO = [NIVEL_PRINCIPAL] + NIVEIS_ROBUSTEZ
DIMENSOES_CLASSIF = ["NO_FAT_AGREG", "NO_CGCE_N3"]
DIMENSOES_DESTINO = ["CO_PAIS"]

resultados = pd.concat(
    [calcular_hhi(base_analise, d) for d in DIMENSOES_PRODUTO + DIMENSOES_CLASSIF]
    + [calcular_hhi(base_destino, d) for d in DIMENSOES_DESTINO],
    ignore_index=True,
)

tabela_H    = resultados.pivot(index="CO_ANO", columns="dimensao", values="H")
tabela_Hn   = resultados.pivot(index="CO_ANO", columns="dimensao", values="H_norm")
tabela_CR4  = resultados.pivot(index="CO_ANO", columns="dimensao", values="CR4")
tabela_CR10 = resultados.pivot(index="CO_ANO", columns="dimensao", values="CR10")

print("HHI bruto (H)")
display(tabela_H.round(4))
print("\nHHI normalizado (H*)")
display(tabela_Hn.round(4))
print("\nCR4")
display(tabela_CR4.round(4))
print("\nCR10")
display(tabela_CR10.round(4))

HHI bruto (H)


dimensao,CAP2,CO_NCM,CO_PAIS,NO_CGCE_N3,NO_FAT_AGREG,SH4,SH6
CO_ANO,,,,,,,
2000,0.0401,0.0111,0.0881,0.1605,0.4508,0.0185,0.0128
2001,0.0388,0.0126,0.0841,0.1374,0.4364,0.0189,0.0136
2002,0.0381,0.0123,0.0849,0.1443,0.4208,0.0182,0.0133
2003,0.0401,0.0125,0.0763,0.1478,0.4148,0.0175,0.0133
2004,0.0422,0.0116,0.0682,0.1460,0.4184,0.0172,0.0121
2005,0.0461,0.0114,0.0611,0.1339,0.4272,0.0177,0.0118
2006,0.0453,0.0128,0.0568,0.1336,0.4203,0.0192,0.0131
2007,0.0438,0.0132,0.0523,0.1314,0.4108,0.0191,0.0135
2008,0.0489,0.0183,0.0487,0.1263,0.3927,0.0251,0.0186



HHI normalizado (H*)


dimensao,CAP2,CO_NCM,CO_PAIS,NO_CGCE_N3,NO_FAT_AGREG,SH4,SH6
CO_ANO,,,,,,,
2000,0.0300,0.0109,0.0836,0.1139,0.1762,0.0176,0.0126
2001,0.0287,0.0125,0.0796,0.0895,0.1546,0.0181,0.0133
2002,0.0280,0.0122,0.0805,0.0967,0.1312,0.0174,0.0131
2003,0.0300,0.0124,0.0720,0.1005,0.1222,0.0167,0.0131
2004,0.0321,0.0114,0.0639,0.0986,0.1276,0.0164,0.0119
2005,0.0360,0.0113,0.0570,0.0858,0.1407,0.0169,0.0116
2006,0.0352,0.0126,0.0528,0.0855,0.1305,0.0184,0.0129
2007,0.0338,0.0131,0.0481,0.0831,0.1162,0.0183,0.0133
2008,0.0389,0.0182,0.0444,0.0777,0.0891,0.0243,0.0184



CR4


dimensao,CAP2,CO_NCM,CO_PAIS,NO_CGCE_N3,NO_FAT_AGREG,SH4,SH6
CO_ANO,,,,,,,
2000,0.2914,0.1418,0.4573,0.5979,1.0,0.1937,0.1571
2001,0.2700,0.1659,0.4299,0.5654,1.0,0.1983,0.1668
2002,0.2584,0.1532,0.3970,0.5781,1.0,0.1857,0.1618
2003,0.2880,0.1578,0.4169,0.5882,1.0,0.1817,0.1592
2004,0.3025,0.1516,0.4003,0.5921,1.0,0.1790,0.1517
2005,0.3256,0.1440,0.3828,0.5665,1.0,0.1818,0.1442
2006,0.3226,0.1647,0.3725,0.5672,1.0,0.2049,0.1647
2007,0.3191,0.1687,0.3719,0.5807,1.0,0.1992,0.1687
2008,0.3400,0.2156,0.3697,0.5911,1.0,0.2471,0.2156



CR10


dimensao,CAP2,CO_NCM,CO_PAIS,NO_CGCE_N3,NO_FAT_AGREG,SH4,SH6
CO_ANO,,,,,,,
2000,0.5405,0.2799,0.6669,0.9094,1.0,0.3510,0.2964
2001,0.5487,0.2904,0.6230,0.8860,1.0,0.3625,0.3023
2002,0.5450,0.2829,0.6037,0.8721,1.0,0.3501,0.2966
2003,0.5567,0.2791,0.6146,0.8721,1.0,0.3400,0.2925
2004,0.5715,0.2663,0.5962,0.8785,1.0,0.3391,0.2731
2005,0.6054,0.2748,0.5747,0.8681,1.0,0.3433,0.2764
2006,0.6043,0.2882,0.5597,0.8687,1.0,0.3562,0.2896
2007,0.5904,0.2898,0.5582,0.8748,1.0,0.3563,0.2901
2008,0.6236,0.3411,0.5490,0.8868,1.0,0.4000,0.3412


### Estabilidade de N e correlação entre H e H*

Base da Tabela 4.4 da monografia. Com a exclusão das operações especiais, o N do fator agregado fica constante em 3; o `assert` abaixo interrompe a execução se isso não ocorrer.

In [17]:
linhas = []
for dim, g in resultados.groupby("dimensao"):
    g = g.sort_values("CO_ANO")
    # correlação indefinida se alguma das séries for constante
    if g["H"].std() == 0 or g["H_norm"].std() == 0:
        corr = np.nan
    else:
        corr = np.corrcoef(g["H"], g["H_norm"])[0, 1]
    linhas.append({
        "dimensao": dim,
        "N_min": int(g["N"].min()),
        "N_max": int(g["N"].max()),
        "variacao": g["N"].max() / g["N"].min(),
        "corr_H_Hnorm": corr,
        "s_max_maximo": g["s_max"].max(),
    })

diagnostico = pd.DataFrame(linhas).sort_values("dimensao").reset_index(drop=True)
display(diagnostico.round(5))

indefinidas = diagnostico.loc[diagnostico["corr_H_Hnorm"].isna(), "dimensao"].tolist()
if indefinidas:
    print(f"Correlação indefinida (série constante) em: {indefinidas}")

n_fat = diagnostico.loc[diagnostico["dimensao"] == "NO_FAT_AGREG"]
if len(n_fat):
    assert n_fat["variacao"].iloc[0] == 1.0, (
        "N do fator agregado varia entre os anos; conferir CATEGORIAS_OPERACAO_ESPECIAL"
    )

suspeitas = diagnostico.loc[diagnostico["corr_H_Hnorm"] < 0.99, "dimensao"].tolist()
if suspeitas:
    print(f"\nH* não acompanha H nas dimensões: {suspeitas}")
    print("Verificar quebra no número de categorias.")
else:
    print("\nTodas as dimensões: H e H* descrevem a mesma trajetória.")

dominadas = diagnostico.loc[diagnostico["s_max_maximo"] > LIMIAR_ALERTA_SMAX, "dimensao"].tolist()
if dominadas:
    print(f"\nDimensões em que um único item supera {LIMIAR_ALERTA_SMAX:.0%} em algum ano: {dominadas}")
    print("Esperado em NO_FAT_AGREG e NO_CGCE_N3, que têm poucas categorias.")

,dimensao,N_min,N_max,variacao,corr_H_Hnorm,s_max_maximo
0,CAP2,96,96,1.00000,1.00000,0.17786
1,CO_NCM,6703,8189,1.22169,1.00000,0.15672
2,CO_PAIS,205,245,1.19512,0.99997,0.32407
3,NO_CGCE_N3,19,19,1.00000,1.00000,0.34638
4,NO_FAT_AGREG,3,3,1.00000,1.00000,0.60975
5,SH4,1176,1210,1.02891,1.00000,0.16821
6,SH6,4335,4963,1.14487,1.00000,0.15672



Todas as dimensões: H e H* descrevem a mesma trajetória.

Dimensões em que um único item supera 25% em algum ano: ['CO_PAIS', 'NO_CGCE_N3', 'NO_FAT_AGREG']
Esperado em NO_FAT_AGREG e NO_CGCE_N3, que têm poucas categorias.


## 10. HHI a preços constantes

O `VL_FOB` está em dólares correntes. Para separar a mudança de composição da variação de preços relativos, a quantidade (`KG_LIQUIDO`) de cada registro é reavaliada pelo valor unitário da sua posição SH-4 no ano-base (seção 3.4, item e, da monografia):

1. Vetor em SH-4, nível em que os códigos não mudam com as revisões do Sistema Harmonizado.
2. Ano-base 2017; 2005 e 2010 entram como sensibilidade (seção 11).
3. Só entram no vetor registros do ano-base com pelo menos `MASSA_MIN_KG`.
4. Ficam fora do vetor as posições em que o quilograma não mede quantidade: capítulos 71 e 89 e posições 8802 e 8411. No capítulo 89 há também um erro de registro na fonte (posição 8906, 2022: 24 milhões de toneladas por US$ 2,8 milhões).
5. A cobertura do vetor (parcela do FOB com preço) é calculada ano a ano.

A série em quantidade física (`H_norm_kg`) é calculada como contraprova. O teste do vetor usado anteriormente está na seção 15.

In [18]:
def vetor_precos(df, ano_base, massa_min=MASSA_MIN_KG, nivel=NIVEL_VETOR_PRECOS):
    """Valor unitário de cada posição no ano-base, com a marcação das excluídas por critério.

    Retorna (vu, vu_valido): `vu` é a tabela completa com a coluna `status`;
    `vu_valido` é a série de preços efetivamente usada na reavaliação.
    """
    vu = (df.loc[(df["CO_ANO"] == ano_base) & (df["KG_LIQUIDO"] >= massa_min)]
            .groupby(nivel)[["VL_FOB", "KG_LIQUIDO"]].sum())
    vu["VU"] = vu["VL_FOB"] / vu["KG_LIQUIDO"]
    excluir = vu.index.isin(EXCLUIR_POSICOES) | vu.index.str[:2].isin(EXCLUIR_CAPITULOS)
    vu["status"] = np.where(excluir, "excluída por critério", "no vetor")
    vu_valido = vu.loc[vu["status"] == "no vetor", "VU"]
    return vu, vu_valido


def reavaliar(df, vu_valido, nivel=NIVEL_VETOR_PRECOS):
    """Quantidade física reavaliada aos preços do ano-base."""
    return np.where(df["KG_LIQUIDO"] > 0,
                    df["KG_LIQUIDO"] * df[nivel].map(vu_valido),
                    np.nan)


def cobertura_por_ano(df, coluna="VL_CONST"):
    fob_ano = df.groupby("CO_ANO")["VL_FOB"].sum()
    coberto = (df.loc[df[coluna].notna()].groupby("CO_ANO")["VL_FOB"].sum()
                 .reindex(fob_ano.index, fill_value=0))
    return (coberto / fob_ano).rename("cobertura")


def motivo_fora_do_vetor(df, vu, coluna="VL_CONST", nivel=NIVEL_VETOR_PRECOS):
    """Classifica o valor que ficou sem preço: exclusão por critério, ausência no ano-base ou falta de peso."""
    fora = df.loc[df[coluna].isna(), [nivel, "VL_FOB", "KG_LIQUIDO"]].copy()
    fora["motivo"] = fora[nivel].map(vu["status"]).fillna("ausente ou abaixo da massa mínima no ano-base")
    fora.loc[fora["KG_LIQUIDO"] <= 0, "motivo"] = "sem quilograma registrado"
    return fora


# Ano-base principal
vu_ano_base, vu_valido = vetor_precos(base_analise, ANO_BASE_PRECOS)
base_analise["VL_CONST"] = reavaliar(base_analise, vu_valido)
cobertura_ano = cobertura_por_ano(base_analise)

n_excl = (vu_ano_base["status"] == "excluída por critério").sum()
print(f"Vetor de preços: {NIVEL_VETOR_PRECOS}, ano-base {ANO_BASE_PRECOS}")
print(f"  posições com massa >= {MASSA_MIN_KG:,} kg no ano-base: {len(vu_ano_base):,}")
print(f"  excluídas por critério: {n_excl} | no vetor: {len(vu_valido):,}")

excluidas_ab = vu_ano_base.loc[vu_ano_base["status"] == "excluída por critério"].copy()
excluidas_ab["pct_fob_ano_base"] = 100 * excluidas_ab["VL_FOB"] / base_analise.loc[
    base_analise["CO_ANO"] == ANO_BASE_PRECOS, "VL_FOB"].sum()
print("\n  Posições excluídas por critério (peso no FOB do ano-base):")
for pos, r in excluidas_ab.sort_values("pct_fob_ano_base", ascending=False).iterrows():
    print(f"    {pos}  {r['pct_fob_ano_base']:6.2f}%   US$ {r['VU']:>12,.2f}/kg")

fora = motivo_fora_do_vetor(base_analise, vu_ano_base)
fob_periodo = base_analise["VL_FOB"].sum()
print("\n  Valor sem preço no período, por motivo (% do FOB):")
for m, v in (100 * fora.groupby("motivo")["VL_FOB"].sum() / fob_periodo).sort_values(ascending=False).items():
    print(f"    {m:<48} {v:.2f}%")

print(f"\nCobertura do vetor de preços de {ANO_BASE_PRECOS}:")
print(f"  média do período : {cobertura_ano.mean():.2%}")
print(f"  mínima ({cobertura_ano.idxmin()})    : {cobertura_ano.min():.2%}")
print(f"  máxima ({cobertura_ano.idxmax()})    : {cobertura_ano.max():.2%}")

Vetor de preços: SH4, ano-base 2017
  posições com massa >= 1,000 kg no ano-base: 1,082
  excluídas por critério: 18 | no vetor: 1,064

  Posições excluídas por critério (peso no FOB do ano-base):
    8802    1.71%   US$     1,212.39/kg
    8411    1.41%   US$     9,343.74/kg
    7108    1.31%   US$    37,166.83/kg
    7112    0.08%   US$        23.70/kg
    7103    0.06%   US$         6.28/kg
    7115    0.01%   US$       816.20/kg
    7106    0.01%   US$       490.30/kg
    8903    0.01%   US$        32.79/kg
    7116    0.01%   US$        11.11/kg
    7117    0.00%   US$       130.85/kg
    8901    0.00%   US$         0.39/kg
    8907    0.00%   US$        21.04/kg
    7113    0.00%   US$       248.89/kg
    7110    0.00%   US$        48.17/kg
    8905    0.00%   US$         1.65/kg
    8902    0.00%   US$         7.56/kg
    8906    0.00%   US$         6.65/kg
    7118    0.00%   US$         1.94/kg

  Valor sem preço no período, por motivo (% do FOB):
    excluída por critério    

In [19]:
# Séries nominal, a preços constantes e em quantidade
nominal = calcular_hhi(base_analise, NIVEL_PRINCIPAL, valor="VL_FOB")
constante = calcular_hhi(base_analise.dropna(subset=["VL_CONST"]), NIVEL_PRINCIPAL, valor="VL_CONST")
quantidade = calcular_hhi(base_analise, NIVEL_PRINCIPAL, valor="KG_LIQUIDO")

comparacao = (nominal.merge(constante, on="CO_ANO", suffixes=("_nominal", "_constante"))
                     .merge(quantidade[["CO_ANO", "H_norm", "s_max"]]
                            .rename(columns={"H_norm": "H_norm_kg", "s_max": "s_max_kg"}),
                            on="CO_ANO"))

comparacao["efeito_preco"] = comparacao["H_norm_nominal"] - comparacao["H_norm_constante"]
comparacao = comparacao.merge(cobertura_ano.reset_index(), on="CO_ANO")

colunas = ["CO_ANO", "H_norm_nominal", "H_norm_constante", "efeito_preco",
           "H_norm_kg", "s_max_constante", "cobertura"]
display(comparacao[colunas].round(4))

# Algum item domina a série a preços constantes?
alerta = comparacao.loc[comparacao["s_max_constante"] > LIMIAR_ALERTA_SMAX, "CO_ANO"].tolist()
if alerta:
    print(f"\nAnos com um item acima do limiar na série a preços constantes: {alerta}")
    print("Maiores posições nesses anos:")
    for ano in alerta:
        piores = (base_analise.loc[(base_analise["CO_ANO"] == ano) & base_analise["VL_CONST"].notna()]
                              .groupby(NIVEL_PRINCIPAL)["VL_CONST"].sum()
                              .sort_values(ascending=False).head(3))
        total = piores.sum() / base_analise.loc[base_analise["CO_ANO"] == ano, "VL_CONST"].sum()
        print(f"  {ano}: {list(piores.index)} ({total:.1%} do valor a preços constantes)")
else:
    print(f"\nNenhum ano com item acima de {LIMIAR_ALERTA_SMAX:.0%} na série a preços constantes.")

print("\nefeito_preco > 0: índice nominal acima do índice a preços constantes.")

,CO_ANO,H_norm_nominal,H_norm_constante,efeito_preco,H_norm_kg,s_max_constante,cobertura
0,2000,0.0176,0.0194,-0.0017,0.4336,0.0849,0.9182
1,2001,0.0181,0.0197,-0.0016,0.3491,0.0708,0.9240
2,2002,0.0174,0.0199,-0.0025,0.3399,0.0715,0.9403
3,2003,0.0167,0.0175,-0.0008,0.3169,0.0645,0.9617
4,2004,0.0164,0.0169,-0.0005,0.3345,0.0674,0.9531
5,2005,0.0169,0.0170,-0.0001,0.3376,0.0659,0.9584
6,2006,0.0184,0.0180,0.0004,0.3451,0.0698,0.9637
7,2007,0.0183,0.0187,-0.0004,0.3586,0.0757,0.9578
8,2008,0.0243,0.0208,0.0035,0.3796,0.0826,0.9609
9,2009,0.0277,0.0271,0.0006,0.3627,0.0881,0.9515



Nenhum ano com item acima de 25% na série a preços constantes.

efeito_preco > 0: índice nominal acima do índice a preços constantes.


In [20]:
# Destino a preços constantes. base_destino foi criada antes de VL_CONST,
# por isso a coluna é copiada de base_analise pelo índice.
base_destino["VL_CONST"] = base_analise.loc[base_destino.index, "VL_CONST"]

cobertura_destino = cobertura_por_ano(base_destino)

destino_nominal = calcular_hhi(base_destino, "CO_PAIS", valor="VL_FOB")
destino_const = calcular_hhi(base_destino.dropna(subset=["VL_CONST"]),
                             "CO_PAIS", valor="VL_CONST")
destino_kg = calcular_hhi(base_destino, "CO_PAIS", valor="KG_LIQUIDO")

destino = (destino_nominal[["CO_ANO", "H", "H_norm", "CR4", "s_max"]]
           .rename(columns={"H": "H_nom", "H_norm": "H_norm_nom",
                            "CR4": "CR4_nom", "s_max": "s_max_nom"})
           .merge(destino_const[["CO_ANO", "H_norm", "CR4", "s_max"]]
                  .rename(columns={"H_norm": "H_norm_const", "CR4": "CR4_const",
                                   "s_max": "s_max_const"}), on="CO_ANO")
           .merge(destino_kg[["CO_ANO", "H_norm"]]
                  .rename(columns={"H_norm": "H_norm_kg"}), on="CO_ANO")
           .merge(cobertura_destino.reset_index(), on="CO_ANO"))

destino["efeito_preco"] = destino["H_norm_nom"] - destino["H_norm_const"]

display(destino.round(4))

# Mínimo e máximo do índice de destino
for rot, col in [("nominal", "H_norm_nom"), ("constante", "H_norm_const")]:
    s = destino.set_index("CO_ANO")[col]
    print(f"{rot:<10} mínimo em {int(s.idxmin())} ({s.min():.4f}) | "
          f"máximo em {int(s.idxmax())} ({s.max():.4f}) | {ANO_FIM} = {s.loc[ANO_FIM]:.4f}")


# Maior destino em cada ano
lider = (base_destino.groupby(["CO_ANO", "NO_PAIS"])["VL_FOB"].sum().reset_index()
         .sort_values(["CO_ANO", "VL_FOB"], ascending=[True, False])
         .groupby("CO_ANO").head(1))
lider["participacao"] = lider["VL_FOB"] / lider["CO_ANO"].map(
    base_destino.groupby("CO_ANO")["VL_FOB"].sum())
print("\nMaior destino por ano:")
print(lider[["CO_ANO", "NO_PAIS", "participacao"]].to_string(index=False))

lider.to_csv(os.path.join(PASTA_EXTRATOS, "destino_lider_ano.csv"), index=False)
destino.to_csv(os.path.join(PASTA_EXTRATOS, "destino_series.csv"), index=False)


# Sobreposição entre as exclusões de operações especiais e de destinos residuais
cruz = pd.crosstab(base["OPERACAO_ESPECIAL"], base["PAIS_RESIDUAL"],
                   values=base["VL_FOB"], aggfunc="sum").fillna(0)
cruz.index = ["produto normal", "operação especial"]
cruz.columns = ["destino efetivo", "destino residual"]

print("Valor FOB por combinação de exclusão (US$ bi):")
display((cruz / 1e9).round(2))

print("\nParcela do FOB total:")
display((cruz / fob_total_base).map(lambda v: f"{v:.4%}"))

so_especial = cruz.loc["operação especial", "destino efetivo"]
so_residual = cruz.loc["produto normal", "destino residual"]
ambos = cruz.loc["operação especial", "destino residual"]
excluido_total = so_especial + so_residual + ambos

print(f"\nExcluído só por operação especial : {so_especial/fob_total_base:.4%}")
print(f"Excluído só por destino residual  : {so_residual/fob_total_base:.4%}")
print(f"Excluído pelos dois critérios     : {ambos/fob_total_base:.4%}")
print(f"União das exclusões               : {excluido_total/fob_total_base:.4%}")

cruz.to_csv(os.path.join(PASTA_EXTRATOS, "sobreposicao_exclusoes.csv")) 

,CO_ANO,H_nom,H_norm_nom,CR4_nom,s_max_nom,H_norm_const,CR4_const,s_max_const,H_norm_kg,cobertura,efeito_preco
0,2000,0.0881,0.0836,0.4573,0.2435,0.0711,0.4313,0.2124,0.0522,0.9182,0.0125
1,2001,0.0841,0.0796,0.4299,0.2482,0.0722,0.4205,0.2267,0.0545,0.9240,0.0074
2,2002,0.0849,0.0805,0.3970,0.2588,0.0584,0.3466,0.2092,0.0539,0.9403,0.0221
3,2003,0.0763,0.0720,0.4169,0.2325,0.0601,0.3886,0.2033,0.0600,0.9617,0.0118
4,2004,0.0682,0.0639,0.4003,0.2133,0.0536,0.3773,0.1814,0.0593,0.9531,0.0103
5,2005,0.0611,0.0570,0.3828,0.1943,0.0494,0.3687,0.1643,0.0616,0.9584,0.0076
6,2006,0.0568,0.0528,0.3725,0.1813,0.0497,0.3757,0.1624,0.0786,0.9637,0.0031
7,2007,0.0523,0.0481,0.3719,0.1595,0.0461,0.3740,0.1422,0.0923,0.9578,0.0021
8,2008,0.0487,0.0444,0.3697,0.1388,0.0428,0.3680,0.1244,0.0859,0.9609,0.0016
9,2009,0.0498,0.0456,0.3781,0.1409,0.0446,0.3768,0.1404,0.1972,0.9515,0.0010


nominal    mínimo em 2008 (0.0444) | máximo em 2020 (0.1203) | 2025 = 0.1004
constante  mínimo em 2008 (0.0428) | máximo em 2020 (0.1113) | 2025 = 0.1095

Maior destino por ano:
 CO_ANO        NO_PAIS  participacao
   2000 Estados Unidos      0.243464
   2001 Estados Unidos      0.248168
   2002 Estados Unidos      0.258793
   2003 Estados Unidos      0.232529
   2004 Estados Unidos      0.213337
   2005 Estados Unidos      0.194276
   2006 Estados Unidos      0.181349
   2007 Estados Unidos      0.159533
   2008 Estados Unidos      0.138794
   2009          China      0.140850
   2010          China      0.156283
   2011          China      0.178106
   2012          China      0.175473
   2013          China      0.201956
   2014          China      0.187775
   2015          China      0.191157
   2016          China      0.197947
   2017          China      0.223612
   2018          China      0.278347
   2019          China      0.286532
   2020          China      0.324065
   2021 

,destino efetivo,destino residual
produto normal,5080.51,0.02
operação especial,5.49,49.59



Parcela do FOB total:


,destino efetivo,destino residual
produto normal,98.9271%,0.0004%
operação especial,0.1070%,0.9656%



Excluído só por operação especial : 0.1070%
Excluído só por destino residual  : 0.0004%
Excluído pelos dois critérios     : 0.9656%
União das exclusões               : 1.0729%


### Maior destino a preços constantes

Participação e identidade do maior destino em valores correntes e a preços constantes (Tabelas 4.3 e A.5 da monografia) e ano da troca de liderança em cada série.

In [21]:
# Maior destino e participação da China em valores correntes e a preços constantes.
# A preços constantes, entram só os registros com preço no ano-base.


def participacoes(df, valor):
    """Participação de cada destino no valor do ano."""
    d = (df.loc[df[valor] > 0]
           .groupby(["CO_ANO", "NO_PAIS"], as_index=False)[valor].sum())
    d["s"] = d[valor] / d.groupby("CO_ANO")[valor].transform("sum")
    return d


def maior_por_ano(d, sufixo):
    """Destino de maior participação em cada ano."""
    m = (d.sort_values(["CO_ANO", "s"], ascending=[True, False])
          .groupby("CO_ANO").head(1)[["CO_ANO", "NO_PAIS", "s"]])
    return m.rename(columns={"NO_PAIS": f"maior_{sufixo}", "s": f"s_maior_{sufixo}"})


def eh_china(d):
    # comparação exata após remover acentos: não confunde com Hong Kong ou Macau
    return d["NO_PAIS"].map(sem_acento).str.strip() == "CHINA"


part_cor = participacoes(base_destino, "VL_FOB")
part_con = participacoes(base_destino, "VL_CONST")

china = (part_cor.loc[eh_china(part_cor), ["CO_ANO", "s"]]
                 .rename(columns={"s": "s_china_corrente"})
         .merge(part_con.loc[eh_china(part_con), ["CO_ANO", "s"]]
                        .rename(columns={"s": "s_china_constante"}),
                on="CO_ANO", how="outer"))

maior_destino = (maior_por_ano(part_cor, "corrente")
                 .merge(maior_por_ano(part_con, "constante"), on="CO_ANO")
                 .merge(china, on="CO_ANO", how="left")
                 .sort_values("CO_ANO")
                 .reset_index(drop=True))

colunas_pct = [c for c in maior_destino.columns if c.startswith("s_")]
exibicao = maior_destino.copy()
exibicao[colunas_pct] = (100 * exibicao[colunas_pct]).round(1)
print("Participações em % do valor do ano (constante: parcela coberta pelo vetor de preços)\n")
display(exibicao)

# Conferência com a série de destino
ref = destino.set_index("CO_ANO")
md = maior_destino.set_index("CO_ANO")
dif_cor = (md["s_maior_corrente"] - ref["s_max_nom"]).abs().max()
dif_con = (md["s_maior_constante"] - ref["s_max_const"]).abs().max()
print(f"\nDiferença máxima para a série de destino: corrente {dif_cor:.1e} | constante {dif_con:.1e} "
      "(esperado: zero ou quase)")


# Anos de troca de liderança
print("\nTrocas de liderança:")
for suf in ["corrente", "constante"]:
    col = maior_destino[f"maior_{suf}"]
    trocas = maior_destino.loc[col != col.shift(), ["CO_ANO", f"maior_{suf}"]]
    texto = "; ".join(f"{int(a)}: {p}" for a, p in trocas.itertuples(index=False))
    print(f"  {suf:<10} {texto}")

vale_nom = int(ref["H_norm_nom"].idxmin())
vale_con = int(ref["H_norm_const"].idxmin())
print(f"\nVale do H* de destino: {vale_nom} (corrente) | {vale_con} (constante)")

diverge = maior_destino.loc[maior_destino["maior_corrente"] != maior_destino["maior_constante"]]
if diverge.empty:
    print("\nO maior destino é o mesmo nas duas séries em todos os anos.")
else:
    print("\nO maior destino difere entre as séries nestes anos:")
    display(diverge.assign(**{c: (100 * diverge[c]).round(1) for c in colunas_pct}))

maior_destino.to_csv(os.path.join(PASTA_EXTRATOS, "maior_destino_corrente_constante.csv"),
                     index=False)


Participações em % do valor do ano (constante: parcela coberta pelo vetor de preços)



,CO_ANO,maior_corrente,s_maior_corrente,maior_constante,s_maior_constante,s_china_corrente,s_china_constante
0,2000,Estados Unidos,24.3,Estados Unidos,21.2,2.0,2.4
1,2001,Estados Unidos,24.8,Estados Unidos,22.7,3.3,3.6
2,2002,Estados Unidos,25.9,Estados Unidos,20.9,4.3,4.5
3,2003,Estados Unidos,23.3,Estados Unidos,20.3,6.3,6.1
4,2004,Estados Unidos,21.3,Estados Unidos,18.1,5.8,5.5
5,2005,Estados Unidos,19.4,Estados Unidos,16.4,5.9,6.0
6,2006,Estados Unidos,18.1,Estados Unidos,16.2,6.2,6.9
7,2007,Estados Unidos,16.0,Estados Unidos,14.2,6.9,7.5
8,2008,Estados Unidos,13.9,Estados Unidos,12.4,8.6,8.0
9,2009,China,14.1,China,14.0,14.1,14.0



Diferença máxima para a série de destino: corrente 0.0e+00 | constante 2.8e-17 (esperado: zero ou quase)

Trocas de liderança:
  corrente   2000: Estados Unidos; 2009: China
  constante  2000: Estados Unidos; 2009: China

Vale do H* de destino: 2008 (corrente) | 2008 (constante)

O maior destino é o mesmo nas duas séries em todos os anos.


## 11. Sensibilidade ao ano-base

Repete a reavaliação para 2005, 2010 e 2017, nas dimensões de produto e de destino. Como 2010 coincide com o pico dos preços das commodities, essa base dá peso maior a minério e soja nos anos iniciais. O interesse está em saber se as três bases descrevem a mesma trajetória, já que os níveis dependem do vetor de preços.

In [22]:
series_base, series_dest_base, china_base = [], [], {}
cols_prod = ["CO_ANO", "KG_LIQUIDO", "VL_FOB"] + list(dict.fromkeys([NIVEL_PRINCIPAL, NIVEL_VETOR_PRECOS]))
cols_dest = ["CO_ANO", "CO_PAIS", "NO_PAIS", "KG_LIQUIDO", "VL_FOB", NIVEL_VETOR_PRECOS]

for ano_base in ANOS_BASE_TESTE:
    _, vu_b = vetor_precos(base_analise, ano_base)

    aux = base_analise[cols_prod].copy()
    aux["VL_CONST"] = reavaliar(aux, vu_b)
    cob = cobertura_por_ano(aux)
    r = calcular_hhi(aux.dropna(subset=["VL_CONST"]), NIVEL_PRINCIPAL, valor="VL_CONST")
    r = r.merge(cob.reset_index(), on="CO_ANO")
    r["ano_base"] = ano_base
    series_base.append(r[["CO_ANO", "ano_base", "H_norm", "s_max", "cobertura"]])

    auxd = base_destino[cols_dest].copy()
    auxd["VL_CONST"] = reavaliar(auxd, vu_b)
    rd = calcular_hhi(auxd.dropna(subset=["VL_CONST"]), "CO_PAIS", valor="VL_CONST")
    rd["ano_base"] = ano_base
    series_dest_base.append(rd[["CO_ANO", "ano_base", "H_norm", "s_max"]])

    part_b = participacoes(auxd, "VL_CONST")
    china_base[ano_base] = part_b.loc[eh_china(part_b)].set_index("CO_ANO")["s"]

    print(f"base {ano_base}: {len(vu_b):,} posições no vetor | cobertura média {cob.mean():.1%} | "
          f"s_max máximo (produto) {r['s_max'].max():.3f}")

sensibilidade = pd.concat(series_base, ignore_index=True)
sensibilidade_destino = pd.concat(series_dest_base, ignore_index=True)

painel = sensibilidade.pivot(index="CO_ANO", columns="ano_base", values="H_norm")
painel.columns = [f"H*_base{c}" for c in painel.columns]
painel = painel.join(nominal.set_index("CO_ANO")["H_norm"].rename("H*_nominal"))
painel = painel.join(quantidade.set_index("CO_ANO")["H_norm"].rename("H*_quantidade"))

painel_destino = sensibilidade_destino.pivot(index="CO_ANO", columns="ano_base", values="H_norm")
painel_destino.columns = [f"H*_base{c}" for c in painel_destino.columns]
painel_destino = painel_destino.join(destino.set_index("CO_ANO")["H_norm_nom"].rename("H*_nominal"))
for b, s in china_base.items():
    painel_destino[f"china_base{b}"] = s
painel_destino["china_nominal"] = maior_destino.set_index("CO_ANO")["s_china_corrente"]

print("\nProduto: trajetórias por ano-base (nível principal:", NIVEL_PRINCIPAL, ")")
display(painel.round(4))
print("\nProduto: correlação entre as séries")
display(painel.corr().round(4))

print("\nDestino: trajetórias por ano-base e participação da China")
display(painel_destino.round(4))

# Efeito preço sob cada ano-base: o sinal muda?
efeitos = pd.DataFrame({f"efeito_base{b}": painel["H*_nominal"] - painel[f"H*_base{b}"]
                        for b in ANOS_BASE_TESTE})
efeitos_destino = pd.DataFrame({f"efeito_base{b}": painel_destino["H*_nominal"] - painel_destino[f"H*_base{b}"]
                                for b in ANOS_BASE_TESTE})
print("\nEfeito preço no produto (H* nominal menos H* a preços constantes)")
display(efeitos.round(4))

for rot, ef in [("produto", efeitos), ("destino", efeitos_destino)]:
    sinais = np.sign(ef).nunique(axis=1)
    robustos = sinais[sinais == 1].index.tolist()
    print(f"\n{rot}: anos com sinal do efeito preço igual nas três bases: {robustos}")

# Resultados centrais sob as três bases
anos_max_prod = {b: int(painel[f"H*_base{b}"].idxmax()) for b in ANOS_BASE_TESTE}
vales_dest = {b: int(painel_destino[f"H*_base{b}"].idxmin()) for b in ANOS_BASE_TESTE}
anos_max_dest = {b: int(painel_destino[f"H*_base{b}"].idxmax()) for b in ANOS_BASE_TESTE}
smax_max = sensibilidade["s_max"].max()

print("\nResultados centrais sob as três bases:")
checks = [
    ("Máximo do produto a preços constantes em 2025", all(a == ANO_FIM for a in anos_max_prod.values())),
    ("Máximo do produto em valores correntes em 2021", int(painel["H*_nominal"].idxmax()) == 2021),
    ("Vale do destino em 2008 ou 2009", all(a in (2008, 2009) for a in vales_dest.values())),
    ("Máximo do destino em 2020", all(a == 2020 for a in anos_max_dest.values())),
    (f"Nenhuma posição acima de {LIMIAR_ALERTA_SMAX:.0%} da série de produto", smax_max <= LIMIAR_ALERTA_SMAX),
]
for texto, ok in checks:
    print(f"   {'OK ' if ok else 'NÃO'}  {texto}")
print(f"\n   anos de máximo do produto por base: {anos_max_prod}")
print(f"   vale e máximo do destino por base : {vales_dest} | {anos_max_dest}")

base 2005: 1,095 posições no vetor | cobertura média 95.6% | s_max máximo (produto) 0.149
base 2010: 1,067 posições no vetor | cobertura média 95.6% | s_max máximo (produto) 0.165
base 2017: 1,064 posições no vetor | cobertura média 95.6% | s_max máximo (produto) 0.154

Produto: trajetórias por ano-base (nível principal: SH4 )


,H*_base2005,H*_base2010,H*_base2017,H*_nominal,H*_quantidade
CO_ANO,,,,,
2000,0.0190,0.0318,0.0194,0.0176,0.4336
2001,0.0195,0.0293,0.0197,0.0181,0.3491
2002,0.0203,0.0293,0.0199,0.0174,0.3399
2003,0.0182,0.0255,0.0175,0.0167,0.3169
2004,0.0173,0.0253,0.0169,0.0164,0.3345
2005,0.0175,0.0252,0.0170,0.0169,0.3376
2006,0.0186,0.0273,0.0180,0.0184,0.3451
2007,0.0194,0.0292,0.0187,0.0183,0.3586
2008,0.0215,0.0330,0.0208,0.0243,0.3796



Produto: correlação entre as séries


,H*_base2005,H*_base2010,H*_base2017,H*_nominal,H*_quantidade
H*_base2005,1.0000,0.9651,0.9933,0.8896,-0.6969
H*_base2010,0.9651,1.0000,0.9709,0.8917,-0.5068
H*_base2017,0.9933,0.9709,1.0000,0.8994,-0.6536
H*_nominal,0.8896,0.8917,0.8994,1.0000,-0.5499
H*_quantidade,-0.6969,-0.5068,-0.6536,-0.5499,1.0000



Destino: trajetórias por ano-base e participação da China


,H*_base2005,H*_base2010,H*_base2017,H*_nominal,china_base2005,china_base2010,china_base2017,china_nominal
CO_ANO,,,,,,,,
2000,0.0726,0.0679,0.0711,0.0836,0.0247,0.0300,0.0240,0.0201
2001,0.0686,0.0631,0.0722,0.0796,0.0380,0.0463,0.0356,0.0334
2002,0.0614,0.0575,0.0584,0.0805,0.0465,0.0559,0.0451,0.0426
2003,0.0615,0.0575,0.0601,0.0720,0.0657,0.0724,0.0606,0.0633
2004,0.0564,0.0521,0.0536,0.0639,0.0586,0.0687,0.0555,0.0580
2005,0.0512,0.0479,0.0494,0.0570,0.0622,0.0740,0.0601,0.0586
2006,0.0504,0.0486,0.0497,0.0528,0.0696,0.0858,0.0690,0.0622
2007,0.0467,0.0460,0.0461,0.0481,0.0756,0.0959,0.0752,0.0687
2008,0.0433,0.0428,0.0428,0.0444,0.0812,0.0997,0.0805,0.0865



Efeito preço no produto (H* nominal menos H* a preços constantes)


,efeito_base2005,efeito_base2010,efeito_base2017
CO_ANO,,,
2000,-0.0013,-0.0142,-0.0017
2001,-0.0014,-0.0112,-0.0016
2002,-0.0029,-0.0119,-0.0025
2003,-0.0015,-0.0088,-0.0008
2004,-0.0009,-0.0089,-0.0005
2005,-0.0006,-0.0083,-0.0001
2006,-0.0003,-0.0089,0.0004
2007,-0.0011,-0.0109,-0.0004
2008,0.0027,-0.0087,0.0035



produto: anos com sinal do efeito preço igual nas três bases: [2000, 2001, 2002, 2003, 2004, 2005, 2007, 2011, 2015, 2016, 2017, 2021, 2022, 2025]

destino: anos com sinal do efeito preço igual nas três bases: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2011, 2013, 2021, 2024, 2025]

Resultados centrais sob as três bases:
   OK   Máximo do produto a preços constantes em 2025
   OK   Máximo do produto em valores correntes em 2021
   OK   Vale do destino em 2008 ou 2009
   OK   Máximo do destino em 2020
   OK   Nenhuma posição acima de 25% da série de produto

   anos de máximo do produto por base: {2005: 2025, 2010: 2025, 2017: 2025}
   vale e máximo do destino por base : {2005: 2008, 2010: 2008, 2017: 2008} | {2005: 2020, 2010: 2020, 2017: 2020}


## 12. Extratos

Arquivos pequenos para consulta fora do notebook.

In [23]:
# 1. Vetor de preços do ano-base, com o status de cada posição
(vu_ano_base.sort_values("VU", ascending=False)
            .to_csv(os.path.join(PASTA_EXTRATOS, f"vetor_precos_{NIVEL_VETOR_PRECOS}_{ANO_BASE_PRECOS}.csv")))

# 2. 25 maiores posições SH-4 por ano
tot_ano = base_analise.groupby("CO_ANO")["VL_FOB"].sum()

top_principal = (base_analise.groupby(["CO_ANO", NIVEL_PRINCIPAL])["VL_FOB"]
                 .sum().reset_index())
top_principal["participacao"] = (top_principal["VL_FOB"] /
                                 top_principal["CO_ANO"].map(tot_ano))
top_principal = (top_principal.sort_values(["CO_ANO", "VL_FOB"], ascending=[True, False])
                              .groupby("CO_ANO").head(25))

# o top 10 deve reproduzir o CR10 da seção 9
_chk = top_principal.groupby("CO_ANO")["participacao"].apply(lambda s: s.nlargest(10).sum())
assert np.allclose(_chk.values, tabela_CR10[NIVEL_PRINCIPAL].reindex(_chk.index).values, atol=1e-6), \
    "Participações do extrato diferentes do CR10"

top_principal.to_csv(os.path.join(PASTA_EXTRATOS, "top25_sh4.csv"), index=False)

# 3. FOB por país e ano, com a marcação de residual preservada
(base.groupby(["CO_ANO", "NO_PAIS", "PAIS_RESIDUAL"])["VL_FOB"].sum().reset_index()
     .to_csv(os.path.join(PASTA_EXTRATOS, "fob_pais_ano.csv"), index=False))

# 4. FOB por fator agregado e ano
(base.groupby(["CO_ANO", "NO_FAT_AGREG"], dropna=False)["VL_FOB"].sum().reset_index()
     .to_csv(os.path.join(PASTA_EXTRATOS, "fob_fatagreg_ano.csv"), index=False))

for nome in sorted(os.listdir(PASTA_EXTRATOS)):
    tam = os.path.getsize(os.path.join(PASTA_EXTRATOS, nome)) / 1024
    print(f"{nome:<28} {tam:>8.1f} KB")

cobertura_china.csv               1.1 KB
destino_lider_ano.csv             1.2 KB
destino_series.csv                5.3 KB
diagnostico_8906.csv              2.4 KB
diagnostico_vetor_precos.csv      6.1 KB
fob_fatagreg_ano.csv              4.5 KB
fob_pais_ano.csv                189.6 KB
maior_destino_corrente_constante.csv      2.7 KB
sensibilidade_sh4_criterio_resumo.csv      0.4 KB
sensibilidade_sh4_criterio_series.csv      5.9 KB
series_sh4_criterio_base_principal.csv      6.0 KB
sobreposicao_exclusoes.csv        0.1 KB
teste_sh4_com_sem_corte.csv       6.1 KB
teste_sh4_posicoes_excluidas.csv    110.0 KB
teste_sh4_sensibilidade_produto.csv      4.5 KB
teste_sh4_sensibilidade_resumo.csv      0.8 KB
top25_sh4.csv                    27.1 KB
vetor_precos_SH4_2017.csv        52.6 KB
vu_diagnostico.csv                9.4 KB


## 13. Planilha de resultados

In [24]:
caminho_xlsx = os.path.join(PASTA_SAIDA, "resultados_hhi.xlsx")

with pd.ExcelWriter(caminho_xlsx, engine="openpyxl") as xl:
    resultados.to_excel(xl, sheet_name="hhi_longo", index=False)
    tabela_H.round(6).to_excel(xl, sheet_name="hhi_bruto")
    tabela_Hn.round(6).to_excel(xl, sheet_name="hhi_normalizado")
    tabela_CR4.round(6).to_excel(xl, sheet_name="cr4")
    tabela_CR10.round(6).to_excel(xl, sheet_name="cr10")
    comparacao.round(6).to_excel(xl, sheet_name="efeito_preco", index=False)
    painel.round(6).to_excel(xl, sheet_name="sensibilidade_base")
    efeitos.round(6).to_excel(xl, sheet_name="sensibilidade_efeito")
    diagnostico.round(6).to_excel(xl, sheet_name="diagnostico_N", index=False)
    resumo_residual.reset_index().to_excel(xl, sheet_name="paises_excluidos", index=False)
    resumo_especial.to_excel(xl, sheet_name="operacoes_especiais")
    val.to_excel(xl, sheet_name="validacao_fob", index=False)
    if lin is not None:
        lin.to_excel(xl, sheet_name="validacao_linhas", index=False)
    pd.Series(numeros_texto).to_frame("valor").to_excel(xl, sheet_name="numeros_texto")
    auditoria.to_excel(xl, sheet_name="auditoria_ingestao", index=False)
    destino.round(6).to_excel(xl, sheet_name="destino_series", index=False)
    maior_destino.round(6).to_excel(xl, sheet_name="maior_destino", index=False)
    painel_destino.round(6).to_excel(xl, sheet_name="sensibilidade_destino")
    efeitos_destino.round(6).to_excel(xl, sheet_name="sensibilidade_efeito_dest")
    vu_ano_base.round(6).to_excel(xl, sheet_name="vetor_precos")

print(f"Resultados em: {caminho_xlsx}")
print(f"Base tratada em: {caminho_parquet}")
print(f"Extratos em: {PASTA_EXTRATOS}")
print(f"Números do texto em: {caminho_numeros}")

Resultados em: G:\My Drive\Puc\8 semestre\Monografia\Segundo Tema\dados_tratados\resultados_hhi.xlsx
Base tratada em: G:\My Drive\Puc\8 semestre\Monografia\Segundo Tema\dados_tratados\base_agregada.csv.gz
Extratos em: G:\My Drive\Puc\8 semestre\Monografia\Segundo Tema\dados_tratados\extratos
Números do texto em: G:\My Drive\Puc\8 semestre\Monografia\Segundo Tema\dados_tratados\numeros_para_o_texto.csv


## 14. Série corrente no universo do vetor, testes e comparações externas

- Série em valores correntes calculada só sobre os registros com preço no ano-base. É ela que se compara com a série a preços constantes no critério de corroboração (seção 3.4, itens e e f), e o efeito preço da monografia é medido contra ela.
- Teste de sinal das proposições, com deslocamento das datas (Tabela 4.9).
- Testes de tendência e de quebra estrutural (seção 3.4, item h; Tabela 4.10).
- Comparação com o índice de concentração da UNCTAD (seção 4.4.5).
- Correlação do efeito preço com o índice de preços de commodities do FMI (seção 3.4, item g).

As séries da UNCTAD e do FMI estão transcritas no código, com fonte e data de consulta. Os resultados são gravados em `resultados_complementares.xlsx`.

In [25]:
# Série corrente no universo do vetor, sob as três bases, e efeito preço (seção 3.4, item e)

cols_prod_u = ["CO_ANO", "KG_LIQUIDO", "VL_FOB"] + list(dict.fromkeys([NIVEL_PRINCIPAL, NIVEL_VETOR_PRECOS]))
cols_dest_u = ["CO_ANO", "CO_PAIS", "NO_PAIS", "KG_LIQUIDO", "VL_FOB", NIVEL_VETOR_PRECOS]

univ_prod, univ_dest, china_univ, cobert = [], [], {}, {}
for ano_base in ANOS_BASE_TESTE:
    _, vu_b = vetor_precos(base_analise, ano_base)

    aux = base_analise[cols_prod_u].copy()
    aux["VL_CONST"] = reavaliar(aux, vu_b)
    no_vetor = aux.loc[aux["VL_CONST"].notna()]
    r = calcular_hhi(no_vetor, NIVEL_PRINCIPAL, valor="VL_FOB")[["CO_ANO", "H_norm", "CR4", "CR10"]]
    r["ano_base"] = ano_base
    univ_prod.append(r)
    cobert[ano_base] = cobertura_por_ano(aux)

    auxd = base_destino[cols_dest_u].copy()
    auxd["VL_CONST"] = reavaliar(auxd, vu_b)
    no_vetor_d = auxd.loc[auxd["VL_CONST"].notna()]
    rd = calcular_hhi(no_vetor_d, "CO_PAIS", valor="VL_FOB")[["CO_ANO", "H_norm"]]
    rd["ano_base"] = ano_base
    univ_dest.append(rd)

    pc = participacoes(no_vetor_d, "VL_FOB")
    china_univ[ano_base] = pc.loc[eh_china(pc)].set_index("CO_ANO")["s"]

univ_prod = pd.concat(univ_prod).pivot(index="CO_ANO", columns="ano_base", values="H_norm")
univ_prod.columns = [f"H*_corrente_universo_base{c}" for c in univ_prod.columns]
univ_dest = pd.concat(univ_dest).pivot(index="CO_ANO", columns="ano_base", values="H_norm")
univ_dest.columns = [f"H*_corrente_universo_base{c}" for c in univ_dest.columns]

efeito_prod_u = pd.DataFrame({f"efeito_base{b}": univ_prod[f"H*_corrente_universo_base{b}"] - painel[f"H*_base{b}"]
                              for b in ANOS_BASE_TESTE})
efeito_dest_u = pd.DataFrame({f"efeito_base{b}": univ_dest[f"H*_corrente_universo_base{b}"] - painel_destino[f"H*_base{b}"]
                              for b in ANOS_BASE_TESTE})

china_univ = pd.DataFrame({f"china_corrente_universo_base{b}": s for b, s in china_univ.items()})

# Tabela A.1 e participações por fator agregado (Tabela A.6)
tab41 = pd.DataFrame({
    "valor_fob_usd_bi": base.groupby("CO_ANO")["VL_FOB"].sum() / 1e9,
    "posicoes_SH4_ativas": tabela_H.index.map(resultados.loc[resultados["dimensao"] == NIVEL_PRINCIPAL]
                                              .set_index("CO_ANO")["N"]),
    "destinos_ativos": tabela_H.index.map(resultados.loc[resultados["dimensao"] == "CO_PAIS"]
                                          .set_index("CO_ANO")["N"]),
})
fat = (base_analise.dropna(subset=["NO_FAT_AGREG"])
                   .groupby(["CO_ANO", "NO_FAT_AGREG"])["VL_FOB"].sum().unstack())
fat_pct = 100 * fat.div(fat.sum(axis=1), axis=0)

display(univ_prod.join(painel).round(4))
display(univ_dest.join(painel_destino).round(4))
display(efeito_prod_u.round(4))
display(efeito_dest_u.round(4))

,H*_corrente_universo_base2005,H*_corrente_universo_base2010,H*_corrente_universo_base2017,H*_base2005,H*_base2010,H*_base2017,H*_nominal,H*_quantidade
CO_ANO,,,,,,,,
2000,0.0159,0.0160,0.0160,0.0190,0.0318,0.0194,0.0176,0.4336
2001,0.0170,0.0171,0.0171,0.0195,0.0293,0.0197,0.0181,0.3491
2002,0.0172,0.0172,0.0173,0.0203,0.0293,0.0199,0.0174,0.3399
2003,0.0171,0.0172,0.0172,0.0182,0.0255,0.0175,0.0167,0.3169
2004,0.0166,0.0166,0.0166,0.0173,0.0253,0.0169,0.0164,0.3345
2005,0.0174,0.0174,0.0175,0.0175,0.0252,0.0170,0.0169,0.3376
2006,0.0190,0.0190,0.0191,0.0186,0.0273,0.0180,0.0184,0.3451
2007,0.0189,0.0188,0.0189,0.0194,0.0292,0.0187,0.0183,0.3586
2008,0.0253,0.0253,0.0253,0.0215,0.0330,0.0208,0.0243,0.3796


,H*_corrente_universo_base2005,H*_corrente_universo_base2010,H*_corrente_universo_base2017,H*_base2005,H*_base2010,H*_base2017,H*_nominal,china_base2005,china_base2010,china_base2017,china_nominal
CO_ANO,,,,,,,,,,,
2000,0.0740,0.0737,0.0740,0.0726,0.0679,0.0711,0.0836,0.0247,0.0300,0.0240,0.0201
2001,0.0682,0.0677,0.0681,0.0686,0.0631,0.0722,0.0796,0.0380,0.0463,0.0356,0.0334
2002,0.0672,0.0669,0.0671,0.0614,0.0575,0.0584,0.0805,0.0465,0.0559,0.0451,0.0426
2003,0.0637,0.0636,0.0637,0.0615,0.0575,0.0601,0.0720,0.0657,0.0724,0.0606,0.0633
2004,0.0563,0.0563,0.0564,0.0564,0.0521,0.0536,0.0639,0.0586,0.0687,0.0555,0.0580
2005,0.0526,0.0526,0.0527,0.0512,0.0479,0.0494,0.0570,0.0622,0.0740,0.0601,0.0586
2006,0.0505,0.0505,0.0506,0.0504,0.0486,0.0497,0.0528,0.0696,0.0858,0.0690,0.0622
2007,0.0460,0.0460,0.0461,0.0467,0.0460,0.0461,0.0481,0.0756,0.0959,0.0752,0.0687
2008,0.0431,0.0430,0.0431,0.0433,0.0428,0.0428,0.0444,0.0812,0.0997,0.0805,0.0865


,efeito_base2005,efeito_base2010,efeito_base2017
CO_ANO,,,
2000,-0.0030,-0.0158,-0.0034
2001,-0.0025,-0.0122,-0.0026
2002,-0.0031,-0.0120,-0.0026
2003,-0.0011,-0.0084,-0.0003
2004,-0.0007,-0.0087,-0.0003
2005,-0.0001,-0.0077,0.0005
2006,0.0004,-0.0082,0.0011
2007,-0.0005,-0.0104,0.0002
2008,0.0038,-0.0077,0.0046


,efeito_base2005,efeito_base2010,efeito_base2017
CO_ANO,,,
2000,0.0015,0.0058,0.0029
2001,-0.0004,0.0046,-0.0042
2002,0.0058,0.0095,0.0088
2003,0.0021,0.0061,0.0036
2004,-0.0002,0.0042,0.0028
2005,0.0014,0.0047,0.0033
2006,0.0001,0.0019,0.0008
2007,-0.0006,0.0000,0.0001
2008,-0.0002,0.0002,0.0003


In [26]:
# Teste de sinal das proposições, com deslocamento das datas (Tabela 4.9)
def variacao(s, a, b):
    return s.loc[b] / s.loc[a] - 1


DATAS = {
    "P1: elevação do produto": [(i, f) for i in (2001, 2003, 2005) for f in (2023, 2025)],
    "P2: queda do destino": [(i, f) for i in (2000, 2002) for f in (2006, 2008, 2010)],
    "P2: elevação do destino": [(i, f) for i in (2006, 2008, 2010) for f in (2023, 2025)],
}
CENTRAIS = {"P1: elevação do produto": (2003, 2025),
            "P2: queda do destino": (2000, 2008),
            "P2: elevação do destino": (2008, 2025)}
SINAL_PREVISTO = {"P1: elevação do produto": 1, "P2: queda do destino": -1, "P2: elevação do destino": 1}

series_sinal = {
    "produto": {"corrente": painel["H*_nominal"],
                "corrente_universo": univ_prod["H*_corrente_universo_base2017"],
                **{f"constante_{b}": painel[f"H*_base{b}"] for b in ANOS_BASE_TESTE}},
    "destino": {"corrente": painel_destino["H*_nominal"],
                "corrente_universo": univ_dest["H*_corrente_universo_base2017"],
                **{f"constante_{b}": painel_destino[f"H*_base{b}"] for b in ANOS_BASE_TESTE}},
}

linhas = []
for mov, pares in DATAS.items():
    dim = "produto" if mov.startswith("P1") else "destino"
    for nome, s in series_sinal[dim].items():
        vs = [variacao(s, a, b) for a, b in pares]
        a, b = CENTRAIS[mov]
        linhas.append({"movimento": mov, "serie": nome,
                       "variacao_central_%": 100 * variacao(s, a, b),
                       "min_datas_deslocadas_%": 100 * min(vs),
                       "max_datas_deslocadas_%": 100 * max(vs),
                       "sinal_previsto_em_todas": all(np.sign(v) == SINAL_PREVISTO[mov] for v in vs)})
teste_sinal = pd.DataFrame(linhas)
display(teste_sinal.round(1))

,movimento,serie,variacao_central_%,min_datas_deslocadas_%,max_datas_deslocadas_%,sinal_previsto_em_todas
0,P1: elevação do produto,corrente,193.8,171.1,245.1,True
1,P1: elevação do produto,corrente_universo,205.4,200.9,255.3,True
2,P1: elevação do produto,constante_2005,196.0,167.7,207.7,True
3,P1: elevação do produto,constante_2010,137.1,101.6,140.5,True
4,P1: elevação do produto,constante_2017,192.3,154.8,201.0,True
5,P2: queda do destino,corrente,-46.9,-46.9,-34.4,True
6,P2: queda do destino,corrente_universo,-41.7,-41.7,-23.7,True
7,P2: queda do destino,constante_2005,-40.3,-40.3,-18.0,True
8,P2: queda do destino,constante_2010,-36.9,-36.9,-9.2,True
9,P2: queda do destino,constante_2017,-39.8,-39.8,-14.8,True


In [27]:
# Testes exploratórios de tendência e de quebra estrutural (seção 3.4, item h).
# Especificação fixada antes da execução: log de H*, tendência linear segmentada,
# segmento mínimo de 5 anos, no máximo 2 quebras, datas por minimização global da SQR
# (Bai e Perron, 2003), p-valores por bootstrap paramétrico com erros AR(1) e
# Mann-Kendall com correção de Hamed e Rao (pip install pymannkendall).
import itertools
import pymannkendall as mk

H_MIN, B_REP = 5, 2000
ANOS_T = np.arange(ANO_INI, ANO_FIM + 1)
rng = np.random.default_rng(20261029)


def _sqr(y, brks):
    t = np.arange(len(y), dtype=float)
    edges, s, yhat = [0, *brks, len(y)], 0.0, np.empty_like(y)
    for a, b in zip(edges, edges[1:]):
        X = np.c_[np.ones(b - a), t[a:b]]
        coef, *_ = np.linalg.lstsq(X, y[a:b], rcond=None)
        yhat[a:b] = X @ coef
        s += float(((y[a:b] - yhat[a:b]) ** 2).sum())
    return s, yhat


def _melhor(y, m, h=H_MIN):
    best = (np.inf, ())
    for b in itertools.combinations(range(h, len(y) - h + 1), m):
        if all(b2 - b1 >= h for b1, b2 in zip(b, b[1:])):
            s, _ = _sqr(y, b)
            if s < best[0]:
                best = (s, b)
    return best


def _supF10(y, h=H_MIN):
    n, q = len(y), 2
    s0, _ = _sqr(y, ())
    s1, b = _melhor(y, 1, h)
    return ((s0 - s1) / q) / (s1 / (n - 2 * q)), b


def _supF21(y, h=H_MIN):
    n, q = len(y), 2
    s1, b1 = _melhor(y, 1, h)
    edges, best = [0, *b1, n], -np.inf
    for a, c in zip(edges, edges[1:]):
        if c - a >= 2 * h:
            s_a, _ = _sqr(y[a:c], ())
            s_b, _ = _melhor(y[a:c], 1, h)
            best = max(best, ((s_a - s_b) / q) / (s1 / (n - 3 * q)))
    return best


def _ar1(yhat, resid, n_rep):
    e = resid - resid.mean()
    rho = float(np.clip(np.corrcoef(e[:-1], e[1:])[0, 1], -0.95, 0.95))
    sig = np.std(e[1:] - rho * e[:-1], ddof=1)
    sims = []
    for _ in range(n_rep):
        u = np.empty_like(e)
        u[0] = rng.normal(0, sig / np.sqrt(1 - rho ** 2))
        for i in range(1, len(e)):
            u[i] = rho * u[i - 1] + rng.normal(0, sig)
        sims.append(yhat + u)
    return sims, rho


def testes_quebra(nome, serie):
    y = np.log(np.asarray(serie, dtype=float))
    F10, b1 = _supF10(y)
    _, yhat0 = _sqr(y, ())
    _, yhat1 = _sqr(y, b1)
    sims_c, rho_c = _ar1(yhat0, y - yhat0, B_REP)   # persistência estimada sob a nula (conservador)
    sims_a, rho_a = _ar1(yhat0, y - yhat1, B_REP)   # persistência estimada no modelo com quebra
    p_c = np.mean([_supF10(s)[0] >= F10 for s in sims_c])
    p_a = np.mean([_supF10(s)[0] >= F10 for s in sims_a])
    F21 = _supF21(y)
    sims_2, _ = _ar1(yhat1, y - yhat1, B_REP)
    p21 = np.mean([_supF21(s) >= F21 for s in sims_2])
    _, b2 = _melhor(y, 2)
    t = np.arange(len(y), dtype=float)
    inc = []
    for a, c in zip([0, b1[0]], [b1[0], len(y)]):
        coef, *_ = np.linalg.lstsq(np.c_[np.ones(c - a), t[a:c]], y[a:c], rcond=None)
        inc.append(100 * (np.exp(coef[1]) - 1))
    r = mk.hamed_rao_modification_test(y)
    return {"serie": nome, "inicio_novo_regime": int(ANOS_T[b1[0]]),
            "supF(1|0)": F10, "p_conservador": p_c, "p_rho_alternativa": p_a,
            "datas_h4_h5_h6": "/".join(str(int(ANOS_T[_melhor(y, 1, h)[1][0]])) for h in (4, 5, 6)),
            "var_anual_antes_%": inc[0], "var_anual_depois_%": inc[1],
            "duas_quebras": "; ".join(str(int(ANOS_T[i])) for i in b2), "supF(2|1)": F21, "p(2|1)": p21,
            "MK_tau": r.Tau, "MK_p": r.p, "Sen_%aa": 100 * (np.exp(r.slope) - 1)}


series_teste = {
    "Produto, constante 2017": painel["H*_base2017"],
    "Produto, corrente no universo do vetor": univ_prod["H*_corrente_universo_base2017"],
    "Produto, corrente sobre o valor total": painel["H*_nominal"],
    "Produto, constante 2005": painel["H*_base2005"],
    "Produto, constante 2010": painel["H*_base2010"],
    "Destino, constante 2017": painel_destino["H*_base2017"],
    "Destino, corrente no universo do vetor": univ_dest["H*_corrente_universo_base2017"],
    "Destino, corrente sobre o valor total": painel_destino["H*_nominal"],
}
quebras = pd.DataFrame([testes_quebra(k, v.loc[ANO_INI:ANO_FIM].values) for k, v in series_teste.items()])
display(quebras.round(3))

,serie,inicio_novo_regime,supF(1|0),p_conservador,p_rho_alternativa,datas_h4_h5_h6,var_anual_antes_%,var_anual_depois_%,duas_quebras,supF(2|1),p(2|1),MK_tau,MK_p,Sen_%aa
0,"Produto, constante 2017",2009,16.814,0.134,0.008,2009/2009/2009,-0.192,4.386,2008; 2019,3.246,0.241,0.846,0.000,5.147
1,"Produto, corrente no universo do vetor",2010,14.785,0.238,0.086,2010/2010/2010,5.671,2.346,2007; 2012,4.283,0.182,0.760,0.000,6.009
2,"Produto, corrente sobre o valor total",2010,10.926,0.401,0.232,2010/2010/2010,4.145,2.397,2007; 2012,4.221,0.216,0.711,0.000,5.624
3,"Produto, constante 2005",2009,14.624,0.155,0.023,2009/2009/2009,0.464,4.661,2008; 2021,2.186,0.584,0.834,0.000,5.056
4,"Produto, constante 2010",2009,19.832,0.170,0.004,2009/2009/2009,-0.034,2.226,2006; 2011,2.206,0.574,0.772,0.000,3.567
5,"Destino, constante 2017",2008,49.720,0.028,0.000,2008/2008/2008,-6.364,6.124,2009; 2018,5.574,0.027,0.502,0.036,3.556
6,"Destino, corrente no universo do vetor",2009,29.045,0.122,0.004,2009/2009/2009,-6.583,6.106,2009; 2018,7.037,0.004,0.446,0.070,3.521
7,"Destino, corrente sobre o valor total",2010,43.227,0.066,0.000,2010/2010/2010,-7.664,5.596,2009; 2018,6.864,0.005,0.372,0.110,2.913


In [28]:
# Índice de concentração de produtos da UNCTAD para o Brasil (exportações).
# Fonte: UNCTADstat, US.ConcentDiversIndices, versão 10007 (atualizada em 24/06/2026),
# consultada em 29/10/2026. Classificação SITC Rev. 3 a três dígitos; normalização por raiz quadrada:
# H = (sqrt(soma s^2) - sqrt(1/n)) / (1 - sqrt(1/n)).
UNCTAD_CI = {1995: 0.086546, 1996: 0.087137, 1997: 0.091867, 1998: 0.089479, 1999: 0.086319,
             2000: 0.088258, 2001: 0.090033, 2002: 0.087899, 2003: 0.084822, 2004: 0.084080,
             2005: 0.086546, 2006: 0.090179, 2007: 0.089780, 2008: 0.108657, 2009: 0.118646,
             2010: 0.153912, 2011: 0.171769, 2012: 0.148838, 2013: 0.157163, 2014: 0.150363,
             2015: 0.131841, 2016: 0.130948, 2017: 0.148155, 2018: 0.170667, 2019: 0.165282,
             2020: 0.182308, 2021: 0.205672, 2022: 0.183946, 2023: 0.196266, 2024: 0.184727,
             2025: 0.178510}
UNCTAD_N = {1995: 241, 1996: 241, 1997: 243, 1998: 241, 1999: 246, 2000: 246, 2001: 245, 2002: 245,
            2003: 246, 2004: 249, 2005: 251, 2006: 249, 2007: 252, 2008: 253, 2009: 251, 2010: 251,
            2011: 251, 2012: 251, 2013: 253, 2014: 253, 2015: 252, 2016: 251, 2017: 249, 2018: 249,
            2019: 248, 2020: 249, 2021: 249, 2022: 251, 2023: 251, 2024: 249, 2025: 250}

sh4 = resultados.loc[resultados["dimensao"] == NIVEL_PRINCIPAL].set_index("CO_ANO")
unctad = pd.DataFrame({
    "H*_linear_SH4": sh4["H_norm"],
    "H*_raiz_SH4": (np.sqrt(sh4["H"]) - np.sqrt(1 / sh4["N"])) / (1 - np.sqrt(1 / sh4["N"])),
    "UNCTAD": pd.Series(UNCTAD_CI),
    "N_UNCTAD": pd.Series(UNCTAD_N),
}).loc[ANO_INI:ANO_FIM]
for col in ["H*_raiz_SH4", "H*_linear_SH4"]:
    print(f"{col}: correlação em nível {unctad[col].corr(unctad['UNCTAD']):.3f} | "
          f"em primeiras diferenças {unctad[col].diff().corr(unctad['UNCTAD'].diff()):.3f}")
display(unctad.round(4))

H*_raiz_SH4: correlação em nível 0.999 | em primeiras diferenças 0.998
H*_linear_SH4: correlação em nível 0.999 | em primeiras diferenças 0.994


,H*_linear_SH4,H*_raiz_SH4,UNCTAD,N_UNCTAD
2000,0.0176,0.1100,0.0883,246
2001,0.0181,0.1117,0.0900,245
2002,0.0174,0.1092,0.0879,245
2003,0.0167,0.1065,0.0848,246
2004,0.0164,0.1052,0.0841,249
2005,0.0169,0.1073,0.0865,251
2006,0.0184,0.1129,0.0902,249
2007,0.0183,0.1127,0.0898,252
2008,0.0243,0.1333,0.1087,253
2009,0.0277,0.1441,0.1186,251


In [29]:
# Índice global de preços de todas as commodities do FMI (PALLFNF, 2016 = 100), média anual.
# Fonte: FMI, Primary Commodity Prices, via FRED (série PALLFNFINDEXA), consultada em 29/10/2026.
FMI = {2000: 60.5748, 2001: 56.3253, 2002: 55.8885, 2003: 65.6735, 2004: 79.1968, 2005: 97.6777,
       2006: 114.0730, 2007: 129.1151, 2008: 162.5599, 2009: 115.9952, 2010: 145.3941, 2011: 181.5763,
       2012: 173.3874, 2013: 168.1630, 2014: 158.2593, 2015: 107.9244, 2016: 100.0000, 2017: 113.3257,
       2018: 127.7253, 2019: 117.0225, 2020: 105.8642, 2021: 161.5026, 2022: 215.9171, 2023: 165.6483,
       2024: 164.8226, 2025: 167.2284}
from scipy import stats

dlog_fmi = np.log(pd.Series(FMI)).diff().loc[ANO_INI + 1:ANO_FIM]
linhas = []
for dim, ef in [("produto", efeito_prod_u), ("destino", efeito_dest_u)]:
    for b in ANOS_BASE_TESTE:
        d = ef[f"efeito_base{b}"].diff().loc[ANO_INI + 1:ANO_FIM]
        pr, pp = stats.pearsonr(d, dlog_fmi)
        sr, sp = stats.spearmanr(d, dlog_fmi)
        linhas.append({"dimensao": dim, "ano_base": b, "pearson": pr, "p_pearson": pp,
                       "spearman": sr, "p_spearman": sp, "n": len(d)})
corr_commodities = pd.DataFrame(linhas)
display(corr_commodities.round(4))

,dimensao,ano_base,pearson,p_pearson,spearman,p_spearman,n
0,produto,2005,0.6334,0.0007,0.5169,0.0081,25
1,produto,2010,0.5808,0.0023,0.4415,0.0271,25
2,produto,2017,0.6712,0.0002,0.5485,0.0045,25
3,destino,2005,0.1994,0.3392,0.1323,0.5284,25
4,destino,2010,0.3454,0.0908,0.2777,0.1790,25
5,destino,2017,0.1364,0.5157,0.0954,0.6502,25


In [30]:
# Gravação dos resultados complementares
caminho_comp = os.path.join(PASTA_SAIDA, "resultados_complementares.xlsx")
with pd.ExcelWriter(caminho_comp, engine="openpyxl") as xl:
    univ_prod.join(painel).round(6).to_excel(xl, sheet_name="produto_universo")
    univ_dest.join(painel_destino).round(6).to_excel(xl, sheet_name="destino_universo")
    efeito_prod_u.round(6).to_excel(xl, sheet_name="efeito_produto_universo")
    efeito_dest_u.round(6).to_excel(xl, sheet_name="efeito_destino_universo")
    china_univ.round(6).to_excel(xl, sheet_name="china_universo")
    pd.DataFrame(cobert).round(6).to_excel(xl, sheet_name="cobertura_bases")
    tab41.round(3).to_excel(xl, sheet_name="tabela_4_1")
    fat_pct.round(3).to_excel(xl, sheet_name="fator_agregado_pct")
    teste_sinal.round(4).to_excel(xl, sheet_name="teste_sinal", index=False)
    quebras.round(4).to_excel(xl, sheet_name="testes_quebra", index=False)
    unctad.round(6).to_excel(xl, sheet_name="unctad")
    corr_commodities.round(4).to_excel(xl, sheet_name="correlacao_commodities", index=False)
print(f"Resultados complementares em: {caminho_comp}")

Resultados complementares em: G:\My Drive\Puc\8 semestre\Monografia\Segundo Tema\dados_tratados\resultados_complementares.xlsx


## 15. Registro: teste do vetor de preços anterior

Reproduz o teste que levou à troca do vetor de preços anterior (NCM-8, base 2010, com corte de 0,5% em cada cauda da distribuição de valores unitários) pelo vetor atual. Fica desligado; para rodar, use `RODAR_REGISTRO_VETOR = True`. Nada nesta seção altera as séries das seções anteriores.

- 15.1: cobertura do vetor anterior por destino e quebra da soja (1201) em 2012.
- 15.2: vetor em SH-4 com e sem corte de cauda e sensibilidade ao ano-base.
- 15.3: vetor em SH-4 com exclusões por critério e diagnóstico da posição 8906.

Com o vetor anterior, a série da China a preços constantes cobria de 26% a 45% do valor exportado a ela a partir de 2012.

In [31]:
if RODAR_REGISTRO_VETOR:
    # Reconstrói o vetor anterior (NCM-8, base 2010, corte de cauda de 0,5%) em cópias,
    # apenas para reproduzir o diagnóstico de out/2026. Nada aqui altera as séries oficiais.
    LEG_ANO_BASE, LEG_CORTE = 2010, 0.005

    vu_leg = (base_analise.loc[(base_analise["CO_ANO"] == LEG_ANO_BASE) & (base_analise["KG_LIQUIDO"] >= MASSA_MIN_KG)]
                          .groupby("CO_NCM")[["VL_FOB", "KG_LIQUIDO"]].sum())
    vu_leg["VU"] = vu_leg["VL_FOB"] / vu_leg["KG_LIQUIDO"]
    li, ls = vu_leg["VU"].quantile([LEG_CORTE, 1 - LEG_CORTE])
    vu_leg_valido = vu_leg.loc[vu_leg["VU"].between(li, ls), "VU"]

    cols_leg = ["CO_ANO", "CO_NCM", "SH4", "CO_PAIS", "NO_PAIS", "VL_FOB", "KG_LIQUIDO"]
    ba_leg = base_analise[cols_leg].copy()
    ba_leg["VL_CONST"] = np.where(ba_leg["KG_LIQUIDO"] > 0,
                                  ba_leg["KG_LIQUIDO"] * ba_leg["CO_NCM"].map(vu_leg_valido), np.nan)
    bd_leg = base_destino[cols_leg].copy()
    bd_leg["VL_CONST"] = ba_leg.loc[bd_leg.index, "VL_CONST"]

    const_leg = calcular_hhi(ba_leg.dropna(subset=["VL_CONST"]), NIVEL_PRINCIPAL, valor="VL_CONST")
    comparacao_leg = (comparacao[["CO_ANO", "H_norm_nominal"]]
                      .merge(const_leg[["CO_ANO", "H_norm"]].rename(columns={"H_norm": "H_norm_constante"}), on="CO_ANO")
                      .merge(cobertura_por_ano(ba_leg).reset_index(), on="CO_ANO"))
    dest_leg = calcular_hhi(bd_leg.dropna(subset=["VL_CONST"]), "CO_PAIS", valor="VL_CONST")
    destino_leg = (destino[["CO_ANO", "H_norm_nom"]]
                   .merge(dest_leg[["CO_ANO", "H_norm"]].rename(columns={"H_norm": "H_norm_const"}), on="CO_ANO"))
    part_leg = participacoes(bd_leg, "VL_CONST")
    maior_destino_leg = (part_leg.loc[eh_china(part_leg), ["CO_ANO", "s"]]
                                 .rename(columns={"s": "s_china_constante"}))

    print(f"Vetor antigo reconstruído: {len(vu_leg_valido):,} NCMs | cobertura média "
          f"{comparacao_leg['cobertura'].mean():.1%} | mínima {comparacao_leg['cobertura'].min():.1%}")

### 15.1 Cobertura do vetor anterior e quebra da soja

In [32]:
if RODAR_REGISTRO_VETOR:
    # Cobertura do vetor anterior para a China e os demais destinos, maiores posições
    # sem preço, cobertura da soja (1201) por ano e comparação com um vetor em SH-4.

    ANOS_INSPECAO = [2011, 2012, 2013, 2017, 2018, 2021, 2025]

    # 1) Cobertura: China x demais destinos
    bd = bd_leg.copy()
    bd["coberto"] = bd["VL_CONST"].notna()
    bd["china"] = eh_china(bd)
    cob = (bd.assign(fob_coberto=bd["VL_FOB"].where(bd["coberto"], 0))
             .groupby(["CO_ANO", "china"])[["fob_coberto", "VL_FOB"]].sum())
    cob = (cob["fob_coberto"] / cob["VL_FOB"]).unstack("china")
    cob.columns = ["cob_demais", "cob_china"]
    print("1) Cobertura do vetor de 2010 (% do FOB de cada grupo)\n")
    display((100 * cob).round(1))

    # 2) Maiores posições sem preço
    print("\n2) Maiores posições SH-4 sem preço no vetor de 2010 (% do FOB do ano)\n")
    ba = ba_leg
    fob_ano = ba.groupby("CO_ANO")["VL_FOB"].sum()
    sem_preco = (ba.loc[ba["VL_CONST"].isna()]
                   .groupby(["CO_ANO", "SH4"])["VL_FOB"].sum().reset_index())
    sem_preco["pct_fob_ano"] = 100 * sem_preco["VL_FOB"] / sem_preco["CO_ANO"].map(fob_ano)
    for ano in ANOS_INSPECAO:
        top = sem_preco.loc[sem_preco["CO_ANO"] == ano].nlargest(5, "VL_FOB")
        itens = ", ".join(f"{r.SH4} ({r.pct_fob_ano:.1f}%)" for r in top.itertuples())
        print(f"  {ano}: {itens}")

    # 3) Soja (1201): cobertura e códigos NCM por ano
    print("\n3) Posição 1201 (soja): parcela do FOB da posição com preço no vetor de 2010\n")
    soja = ba.loc[ba["SH4"] == "1201"]
    cob_soja = (soja.loc[soja["VL_CONST"].notna()].groupby("CO_ANO")["VL_FOB"].sum()
                    .reindex(fob_ano.index, fill_value=0)
                / soja.groupby("CO_ANO")["VL_FOB"].sum())
    ncms_soja = soja.groupby("CO_ANO")["CO_NCM"].unique().map(lambda x: ", ".join(sorted(x)))
    display(pd.DataFrame({"cobertura_1201_%": (100 * cob_soja).round(1),
                          "NCMs exportadas": ncms_soja}))

    # 4) Teste: vetor de preços em SH-4
    def vetor_precos_sh4(df, ano_base, massa_min=MASSA_MIN_KG, corte=LEG_CORTE):
        vu = (df.loc[(df["CO_ANO"] == ano_base) & (df["KG_LIQUIDO"] >= massa_min)]
                .groupby("SH4")[["VL_FOB", "KG_LIQUIDO"]].sum())
        vu["VU"] = vu["VL_FOB"] / vu["KG_LIQUIDO"]
        lim_inf, lim_sup = vu["VU"].quantile([corte, 1 - corte])
        return vu.loc[vu["VU"].between(lim_inf, lim_sup), "VU"]


    vu_sh4 = vetor_precos_sh4(ba_leg, LEG_ANO_BASE)

    def reavaliar_sh4(df):
        return np.where(df["KG_LIQUIDO"] > 0, df["KG_LIQUIDO"] * df["SH4"].map(vu_sh4), np.nan)

    ba4 = ba_leg[["CO_ANO", "SH4", "VL_FOB", "KG_LIQUIDO"]].copy()
    ba4["VL_CONST_SH4"] = reavaliar_sh4(ba4)
    bd4 = bd_leg[["CO_ANO", "SH4", "CO_PAIS", "NO_PAIS", "VL_FOB", "KG_LIQUIDO"]].copy()
    bd4["VL_CONST_SH4"] = reavaliar_sh4(bd4)

    prod_sh4 = calcular_hhi(ba4.dropna(subset=["VL_CONST_SH4"]), NIVEL_PRINCIPAL, valor="VL_CONST_SH4")
    dest_sh4 = calcular_hhi(bd4.dropna(subset=["VL_CONST_SH4"]), "CO_PAIS", valor="VL_CONST_SH4")
    part_sh4 = participacoes(bd4, "VL_CONST_SH4")
    lider_sh4 = maior_por_ano(part_sh4, "const_sh4")
    china_sh4 = (part_sh4.loc[eh_china(part_sh4), ["CO_ANO", "s"]]
                         .rename(columns={"s": "s_china_const_sh4"}))

    comp = (comparacao_leg[["CO_ANO", "H_norm_nominal", "H_norm_constante", "cobertura"]]
              .rename(columns={"H_norm_nominal": "Hp_nominal", "H_norm_constante": "Hp_const_ncm",
                               "cobertura": "cob_ncm"})
              .merge(prod_sh4[["CO_ANO", "H_norm"]].rename(columns={"H_norm": "Hp_const_sh4"}), on="CO_ANO")
              .merge(cobertura_por_ano(ba4, "VL_CONST_SH4").rename("cob_sh4").reset_index(), on="CO_ANO")
              .merge(destino_leg[["CO_ANO", "H_norm_nom", "H_norm_const"]]
                       .rename(columns={"H_norm_nom": "Hd_nominal", "H_norm_const": "Hd_const_ncm"}), on="CO_ANO")
              .merge(dest_sh4[["CO_ANO", "H_norm"]].rename(columns={"H_norm": "Hd_const_sh4"}), on="CO_ANO")
              .merge(maior_destino_leg[["CO_ANO", "s_china_constante"]]
                       .rename(columns={"s_china_constante": "china_const_ncm"}), on="CO_ANO")
              .merge(china_sh4.rename(columns={"s_china_const_sh4": "china_const_sh4"}), on="CO_ANO", how="left")
              .merge(lider_sh4, on="CO_ANO"))

    print("\n4) Séries com vetor em NCM-8 (atual) e em SH-4 (teste). Hp = produto, Hd = destino\n")
    display(comp.round(4))

    for rot, col in [("produto, vetor NCM", "Hp_const_ncm"), ("produto, vetor SH-4", "Hp_const_sh4"),
                     ("destino, vetor NCM", "Hd_const_ncm"), ("destino, vetor SH-4", "Hd_const_sh4")]:
        s = comp.set_index("CO_ANO")[col]
        print(f"  {rot:<22} mínimo {int(s.idxmin())} | máximo {int(s.idxmax())} | "
              f"2025/2000 = {s.loc[2025] / s.loc[2000] - 1:+.1%}")

    col = comp["maior_const_sh4"]
    trocas = comp.loc[col != col.shift(), ["CO_ANO", "maior_const_sh4"]]
    print("\n  Trocas de liderança com vetor SH-4: "
          + "; ".join(f"{int(a)}: {p}" for a, p in trocas.itertuples(index=False)))

    comp.to_csv(os.path.join(PASTA_EXTRATOS, "diagnostico_vetor_precos.csv"), index=False)
    cob.to_csv(os.path.join(PASTA_EXTRATOS, "cobertura_china.csv"))
    print("\nSalvo em extratos: diagnostico_vetor_precos.csv e cobertura_china.csv")


### 15.2 Vetor em SH-4, com e sem corte de cauda

In [33]:
if RODAR_REGISTRO_VETOR:
    # Vetor em SH-4 com e sem corte de cauda: posições que ficam fora, dominância de
    # algum item sem o corte e sensibilidade ao ano-base.

    PRINCIPAIS_SH4 = ["1201", "2601", "2709", "1701", "0202", "0207", "2304", "1005", "0901", "4703"]


    def vetor_sh4_detalhado(df, ano_base, massa_min=MASSA_MIN_KG, corte=LEG_CORTE):
        """Valor unitário por posição SH-4 no ano-base, com o status de cada posição."""
        vu = (df.loc[(df["CO_ANO"] == ano_base) & (df["KG_LIQUIDO"] >= massa_min)]
                .groupby("SH4")[["VL_FOB", "KG_LIQUIDO"]].sum())
        vu["VU"] = vu["VL_FOB"] / vu["KG_LIQUIDO"]
        if corte > 0:
            lim_inf, lim_sup = vu["VU"].quantile([corte, 1 - corte])
        else:
            lim_inf, lim_sup = -np.inf, np.inf
        vu["status"] = np.select([vu["VU"] < lim_inf, vu["VU"] > lim_sup],
                                 ["corte inferior", "corte superior"], "no vetor")
        return vu


    def series_sh4(vu):
        """Séries de produto e de destino reavaliadas com um vetor SH-4."""
        precos = vu.loc[vu["status"] == "no vetor", "VU"]

        pa = base_analise[["CO_ANO", "SH4", "VL_FOB", "KG_LIQUIDO"]].copy()
        pa["VC"] = np.where(pa["KG_LIQUIDO"] > 0, pa["KG_LIQUIDO"] * pa["SH4"].map(precos), np.nan)
        de = base_destino[["CO_ANO", "SH4", "CO_PAIS", "NO_PAIS", "VL_FOB", "KG_LIQUIDO"]].copy()
        de["VC"] = np.where(de["KG_LIQUIDO"] > 0, de["KG_LIQUIDO"] * de["SH4"].map(precos), np.nan)

        hp = calcular_hhi(pa.dropna(subset=["VC"]), NIVEL_PRINCIPAL, valor="VC").set_index("CO_ANO")
        hd = calcular_hhi(de.dropna(subset=["VC"]), "CO_PAIS", valor="VC").set_index("CO_ANO")

        por_pos = pa.groupby(["CO_ANO", "SH4"])["VC"].sum()
        item_max = por_pos.groupby(level=0).idxmax().map(lambda t: t[1])

        part = participacoes(de, "VC")
        china = part.loc[eh_china(part)].set_index("CO_ANO")["s"]
        lider = maior_por_ano(part, "x").set_index("CO_ANO")["maior_x"]

        out = pd.DataFrame({
            "Hp": hp["H_norm"], "smax_prod": hp["s_max"], "item_max": item_max,
            "Hd": hd["H_norm"], "china": china, "lider": lider,
            "cob": cobertura_por_ano(pa, "VC"),
        })
        return out, pa


    def resumo_serie(s):
        return int(s.idxmin()), int(s.idxmax()), s.loc[ANO_FIM] / s.loc[ANO_INI] - 1


    def trocas_lideranca(lider):
        t = lider.loc[lider != lider.shift()]
        return "; ".join(f"{int(a)}: {p}" for a, p in t.items())


    # 1) O que o vetor SH-4 com corte deixa de fora, e por quê
    vu_com = vetor_sh4_detalhado(base_analise, LEG_ANO_BASE, corte=LEG_CORTE)
    ser_com, pa_com = series_sh4(vu_com)

    fora = pa_com.loc[pa_com["VC"].isna()].copy()
    fora["motivo"] = fora["SH4"].map(vu_com["status"]).fillna(
        "ausente ou abaixo da massa mínima no ano-base")
    fora.loc[fora["KG_LIQUIDO"] <= 0, "motivo"] = "sem quilograma registrado"

    fob_total_periodo = pa_com["VL_FOB"].sum()
    fob_ano = pa_com.groupby("CO_ANO")["VL_FOB"].sum()
    anual = fora.groupby(["SH4", "CO_ANO"])["VL_FOB"].sum().unstack(fill_value=0)
    excluidas = pd.DataFrame({
        "motivo": fora.groupby("SH4")["motivo"].agg(lambda m: m.value_counts().index[0]),
        "pct_fob_periodo": 100 * fora.groupby("SH4")["VL_FOB"].sum() / fob_total_periodo,
        "pct_max_anual": 100 * anual.div(fob_ano, axis=1).max(axis=1),
        "VU_ano_base": vu_com["VU"],
    }).dropna(subset=["motivo"]).sort_values("pct_fob_periodo", ascending=False)

    print(f"1) Vetor SH-4 com corte de {LEG_CORTE:.1%}: posições fora da série "
          f"(ano-base {LEG_ANO_BASE})\n")
    print(f"   Posições no vetor: {(vu_com['status'] == 'no vetor').sum()} | "
          f"corte inferior: {(vu_com['status'] == 'corte inferior').sum()} | "
          f"corte superior: {(vu_com['status'] == 'corte superior').sum()}")
    print("   15 maiores, em % do FOB do período e no pior ano:\n")
    display(excluidas.head(15).round({"pct_fob_periodo": 2, "pct_max_anual": 2, "VU_ano_base": 4}))

    cob_pos = (pa_com.assign(coberto=pa_com["VL_FOB"].where(pa_com["VC"].notna(), 0))
                     .loc[pa_com["SH4"].isin(PRINCIPAIS_SH4)]
                     .groupby(["SH4", "CO_ANO"])[["coberto", "VL_FOB"]].sum())
    cob_pos = (cob_pos["coberto"] / cob_pos["VL_FOB"]).unstack()
    falhas = cob_pos.loc[cob_pos.min(axis=1) < 0.95]
    print("\n   Principais posições da pauta com cobertura abaixo de 95% em algum ano:")
    if falhas.empty:
        print("   nenhuma. Soja, minério, petróleo, açúcar, carnes, farelo, milho, café e celulose cobertos.")
    else:
        display((100 * falhas).round(1))

    # 2) Vetor SH-4 sem corte de cauda (só massa mínima)
    vu_sem = vetor_sh4_detalhado(base_analise, LEG_ANO_BASE, corte=0)
    ser_sem, _ = series_sh4(vu_sem)

    comp_corte = pd.concat({"com_corte": ser_com, "sem_corte": ser_sem}, axis=1)
    comp_corte.columns = [f"{b}_{a}" for a, b in comp_corte.columns]
    ordem = ["Hp_com_corte", "Hp_sem_corte", "Hd_com_corte", "Hd_sem_corte",
             "china_com_corte", "china_sem_corte", "cob_com_corte", "cob_sem_corte",
             "smax_prod_sem_corte", "item_max_sem_corte"]
    print(f"\n2) Vetor SH-4 com e sem corte de cauda (ano-base {LEG_ANO_BASE})\n")
    display(comp_corte[ordem].round(4))

    for rot, col in [("produto, com corte", "Hp_com_corte"), ("produto, sem corte", "Hp_sem_corte"),
                     ("destino, com corte", "Hd_com_corte"), ("destino, sem corte", "Hd_sem_corte")]:
        mn, mx, var = resumo_serie(comp_corte[col])
        print(f"  {rot:<20} mínimo {mn} | máximo {mx} | {ANO_FIM}/{ANO_INI} = {var:+.1%}")
    print(f"\n  Liderança, com corte: {trocas_lideranca(ser_com['lider'])}")
    print(f"  Liderança, sem corte: {trocas_lideranca(ser_sem['lider'])}")

    alerta = ser_sem.loc[ser_sem["smax_prod"] > LIMIAR_ALERTA_SMAX]
    if alerta.empty:
        print(f"\n  OK: sem corte, nenhuma posição passa de {LIMIAR_ALERTA_SMAX:.0%} da série de produto "
              f"(máximo: {ser_sem['smax_prod'].max():.1%}, posição "
              f"{ser_sem.loc[ser_sem['smax_prod'].idxmax(), 'item_max']} em {int(ser_sem['smax_prod'].idxmax())}).")
    else:
        print(f"\n  Sem corte, uma posição passa de {LIMIAR_ALERTA_SMAX:.0%} da série de produto:")
        display(alerta[["smax_prod", "item_max"]].round(3))
        print("  Ver o valor unitário dessas posições no ano-base antes de abandonar o corte.")

    # 3) Sensibilidade ao ano-base, nas duas variantes
    linhas_resumo, paineis = [], {}
    for variante, corte in [("com corte", LEG_CORTE), ("sem corte", 0)]:
        hp_bases = {}
        for ano_base in ANOS_BASE_TESTE:
            s, _ = series_sh4(vetor_sh4_detalhado(base_analise, ano_base, corte=corte))
            hp_bases[ano_base] = s["Hp"]
            hp_min, hp_max, hp_var = resumo_serie(s["Hp"])
            hd_min, hd_max, hd_var = resumo_serie(s["Hd"])
            linhas_resumo.append({
                "variante": variante, "ano_base": ano_base,
                "cob_media_%": round(100 * s["cob"].mean(), 1),
                "cob_min_%": round(100 * s["cob"].min(), 1),
                "Hp_ano_max": hp_max, "Hp_var_%": round(100 * hp_var, 1),
                "Hd_vale": hd_min, "Hd_ano_max": hd_max, "Hd_var_%": round(100 * hd_var, 1),
                "china_2020_%": round(100 * s.loc[2020, "china"], 1),
                "troca_lideranca": trocas_lideranca(s["lider"]),
                "smax_prod_max_%": round(100 * s["smax_prod"].max(), 1),
            })
        painel_v = pd.DataFrame({f"base{b}": v for b, v in hp_bases.items()})
        painel_v["nominal"] = comparacao.set_index("CO_ANO")["H_norm_nominal"]
        paineis[variante] = painel_v

    resumo_base = pd.DataFrame(linhas_resumo)
    print("\n3) Sensibilidade ao ano-base com vetor SH-4\n")
    display(resumo_base)

    for variante, painel_v in paineis.items():
        corr = painel_v[[c for c in painel_v if c.startswith("base")]].corr().values
        corr_min = corr[np.triu_indices_from(corr, k=1)].min()
        efeitos_v = pd.DataFrame({b: painel_v["nominal"] - painel_v[b]
                                  for b in painel_v if b.startswith("base")})
        sinal = np.sign(efeitos_v).nunique(axis=1)
        print(f"\n  {variante}: correlação mínima entre as três bases = {corr_min:.3f}")
        print(f"  {variante}: anos em que o sinal do efeito preço depende da base: "
              f"{sinal[sinal > 1].index.tolist() or 'nenhum'}")
        print(f"  {variante}: efeito preço (nominal menos constante) por base, anos selecionados")
        display(efeitos_v.loc[[2000, 2005, 2008, 2011, 2013, 2015, 2020, 2021, 2022, 2024, 2025]].round(4))

    # Salvar
    excluidas.to_csv(os.path.join(PASTA_EXTRATOS, "teste_sh4_posicoes_excluidas.csv"))
    comp_corte.to_csv(os.path.join(PASTA_EXTRATOS, "teste_sh4_com_sem_corte.csv"))
    resumo_base.to_csv(os.path.join(PASTA_EXTRATOS, "teste_sh4_sensibilidade_resumo.csv"), index=False)
    pd.concat(paineis, axis=1).to_csv(os.path.join(PASTA_EXTRATOS, "teste_sh4_sensibilidade_produto.csv"))
    print("\nSalvo em extratos: teste_sh4_posicoes_excluidas.csv, teste_sh4_com_sem_corte.csv,")
    print("teste_sh4_sensibilidade_resumo.csv e teste_sh4_sensibilidade_produto.csv")


### 15.3 Vetor em SH-4 com exclusões por critério

In [34]:
if RODAR_REGISTRO_VETOR:
    # Vetor em SH-4 com exclusões por critério: diagnóstico da posição 8906, cobertura
    # e sensibilidade às três bases. Usa as funções definidas em 15.2.

    # Posições em que o quilograma não mede quantidade: bens de capital unitários e
    # metais e pedras preciosos. Itens sem peso registrado (ex.: 2716, energia) já
    # ficam fora por construção.


    def vetor_sh4_criterio(df, ano_base):
        vu = vetor_sh4_detalhado(df, ano_base, corte=0)
        marcar = vu.index.isin(EXCLUIR_POSICOES) | vu.index.str[:2].isin(EXCLUIR_CAPITULOS)
        vu.loc[marcar, "status"] = "excluída por critério"
        return vu


    # 1) Diagnóstico da posição 8906
    p8906 = base_analise.loc[base_analise["SH4"] == "8906"]
    d8906 = p8906.groupby("CO_ANO")[["VL_FOB", "KG_LIQUIDO"]].sum()
    d8906 = d8906.reindex(fob_ano.index, fill_value=0)
    d8906["pct_fob_ano"] = 100 * d8906["VL_FOB"] / fob_ano
    d8906["mil_toneladas"] = d8906["KG_LIQUIDO"] / 1e6
    d8906["US$_por_kg"] = d8906["VL_FOB"] / d8906["KG_LIQUIDO"].where(d8906["KG_LIQUIDO"] > 0)
    destino_8906 = (base_destino.loc[base_destino["SH4"] == "8906"]
                    .groupby(["CO_ANO", "NO_PAIS"])["VL_FOB"].sum().reset_index()
                    .sort_values(["CO_ANO", "VL_FOB"], ascending=[True, False])
                    .groupby("CO_ANO").head(1).set_index("CO_ANO")["NO_PAIS"])
    d8906["principal_destino"] = destino_8906
    d8906["US$_bi"] = d8906["VL_FOB"] / 1e9

    print("1) Posição 8906 (outras embarcações) ano a ano\n")
    display(d8906[["US$_bi", "pct_fob_ano", "mil_toneladas", "US$_por_kg", "principal_destino"]]
            .round({"US$_bi": 3, "pct_fob_ano": 2, "mil_toneladas": 1, "US$_por_kg": 2}))
    print("\n   Preço por kg da 8906 em cada ano-base testado:")
    for b in ANOS_BASE_TESTE:
        vu_b = vetor_sh4_detalhado(base_analise, b, corte=0)
        v = vu_b["VU"].get("8906", np.nan)
        print(f"   base {b}: US$ {v:,.2f}/kg")

    # 2) Vetor SH-4 com exclusões por critério, ano-base principal
    vu_crit = vetor_sh4_criterio(base_analise, LEG_ANO_BASE)
    ser_crit, pa_crit = series_sh4(vu_crit)

    excl = pa_crit.loc[pa_crit["VC"].isna()].copy()
    excl["motivo"] = excl["SH4"].map(vu_crit["status"]).fillna("ausente ou abaixo da massa mínima no ano-base")
    excl.loc[excl["KG_LIQUIDO"] <= 0, "motivo"] = "sem quilograma registrado"
    fob_periodo = pa_crit["VL_FOB"].sum()
    por_motivo = (100 * excl.groupby("motivo")["VL_FOB"].sum() / fob_periodo).sort_values(ascending=False)

    print(f"\n2) Vetor SH-4 com exclusões por critério (ano-base {LEG_ANO_BASE}, sem corte de cauda)\n")
    print(f"   Cobertura: média {ser_crit['cob'].mean():.1%} | mínima {ser_crit['cob'].min():.1%} "
          f"({int(ser_crit['cob'].idxmin())})")
    print("   Valor fora da série, por motivo (% do FOB do período):")
    for m, v in por_motivo.items():
        print(f"     {m:<48} {v:.2f}%")

    comp_crit = pd.DataFrame({
        "Hp_nominal": comparacao.set_index("CO_ANO")["H_norm_nominal"],
        "Hp_const": ser_crit["Hp"],
        "Hd_nominal": destino.set_index("CO_ANO")["H_norm_nom"],
        "Hd_const": ser_crit["Hd"],
        "china_nominal": maior_destino.set_index("CO_ANO")["s_china_corrente"],
        "china_const": ser_crit["china"],
        "lider_nominal": maior_destino.set_index("CO_ANO")["maior_corrente"],
        "lider_const": ser_crit["lider"],
        "smax_prod_const": ser_crit["smax_prod"],
        "item_max_const": ser_crit["item_max"],
        "cobertura": ser_crit["cob"],
    })
    comp_crit["efeito_preco_prod"] = comp_crit["Hp_nominal"] - comp_crit["Hp_const"]
    comp_crit["efeito_preco_dest"] = comp_crit["Hd_nominal"] - comp_crit["Hd_const"]
    display(comp_crit.round(4))

    for rot, col in [("produto, nominal", "Hp_nominal"), ("produto, constante", "Hp_const"),
                     ("destino, nominal", "Hd_nominal"), ("destino, constante", "Hd_const")]:
        mn, mx, var = resumo_serie(comp_crit[col])
        print(f"  {rot:<20} mínimo {mn} | máximo {mx} | {ANO_FIM}/{ANO_INI} = {var:+.1%}")
    print(f"\n  Liderança, nominal  : {trocas_lideranca(comp_crit['lider_nominal'])}")
    print(f"  Liderança, constante: {trocas_lideranca(comp_crit['lider_const'])}")

    # 3) Sensibilidade às três bases e checagem dos resultados do texto
    series_por_base, linhas = {}, []
    for b in ANOS_BASE_TESTE:
        s, _ = series_sh4(vetor_sh4_criterio(base_analise, b))
        series_por_base[b] = s
        hp_min, hp_max, hp_var = resumo_serie(s["Hp"])
        hd_min, hd_max, hd_var = resumo_serie(s["Hd"])
        linhas.append({
            "ano_base": b, "cob_media_%": round(100 * s["cob"].mean(), 1),
            "cob_min_%": round(100 * s["cob"].min(), 1),
            "Hp_ano_min": hp_min, "Hp_ano_max": hp_max, "Hp_var_%": round(100 * hp_var, 1),
            "Hd_vale": hd_min, "Hd_ano_max": hd_max, "Hd_var_%": round(100 * hd_var, 1),
            "china_2020_%": round(100 * s.loc[2020, "china"], 1),
            "troca_lideranca": trocas_lideranca(s["lider"]),
            "smax_prod_max_%": round(100 * s["smax_prod"].max(), 1),
            "item_smax": s.loc[s["smax_prod"].idxmax(), "item_max"],
        })
    resumo_crit = pd.DataFrame(linhas)
    print("\n3) Sensibilidade ao ano-base, vetor SH-4 com exclusões por critério\n")
    display(resumo_crit)

    painel_crit = pd.DataFrame({f"Hp_base{b}": s["Hp"] for b, s in series_por_base.items()})
    painel_crit["Hp_nominal"] = comp_crit["Hp_nominal"]
    for b, s in series_por_base.items():
        painel_crit[f"Hd_base{b}"] = s["Hd"]
    painel_crit["Hd_nominal"] = comp_crit["Hd_nominal"]
    for b, s in series_por_base.items():
        painel_crit[f"china_base{b}"] = s["china"]

    cols_hp = [f"Hp_base{b}" for b in ANOS_BASE_TESTE]
    cols_hd = [f"Hd_base{b}" for b in ANOS_BASE_TESTE]
    efeito_prod = pd.DataFrame({c: painel_crit["Hp_nominal"] - painel_crit[c] for c in cols_hp})
    efeito_dest = pd.DataFrame({c: painel_crit["Hd_nominal"] - painel_crit[c] for c in cols_hd})


    def corr_min(cols):
        c = painel_crit[cols].corr().values
        return c[np.triu_indices_from(c, k=1)].min()


    def anos_sinal_instavel(ef):
        n = np.sign(ef).nunique(axis=1)
        return n[n > 1].index.tolist()


    print(f"\n  Correlação mínima entre bases: produto {corr_min(cols_hp):.3f} | destino {corr_min(cols_hd):.3f}")
    print(f"  Anos com sinal do efeito preço dependente da base, produto: {anos_sinal_instavel(efeito_prod) or 'nenhum'}")
    print(f"  Anos com sinal do efeito preço dependente da base, destino: {anos_sinal_instavel(efeito_dest) or 'nenhum'}")

    print("\n  Efeito preço por base (nominal menos constante), produto:")
    display(efeito_prod.round(4))

    # Checagem dos resultados que sustentam o texto
    print("\n  Checagem dos resultados centrais nas três bases:")
    checks = [
        ("Máximo do produto a preços constantes em 2025",
         all(r["Hp_ano_max"] == ANO_FIM for r in linhas)),
        ("Máximo do produto em valores correntes em 2021",
         int(comp_crit["Hp_nominal"].idxmax()) == 2021),
        ("Vale do destino em 2008 ou 2009",
         all(r["Hd_vale"] in (2008, 2009) for r in linhas)),
        ("Máximo do destino em 2020",
         all(r["Hd_ano_max"] == 2020 for r in linhas)),
        ("Troca de liderança única, EUA para China, em 2009",
         all(r["troca_lideranca"] == "2000: Estados Unidos; 2009: China" for r in linhas)),
        (f"Nenhuma posição acima de {LIMIAR_ALERTA_SMAX:.0%} da série de produto",
         all(r["smax_prod_max_%"] <= 100 * LIMIAR_ALERTA_SMAX for r in linhas)),
    ]
    for texto, ok in checks:
        print(f"   {'OK       ' if ok else 'NÃO      '} {texto}")
    if not all(ok for _, ok in checks):
        print("\n  Ver a tabela do item 3 para identificar a base e o ano que falham.")

    # Salvar
    d8906.to_csv(os.path.join(PASTA_EXTRATOS, "diagnostico_8906.csv"))
    comp_crit.to_csv(os.path.join(PASTA_EXTRATOS, "series_sh4_criterio_base_principal.csv"))
    resumo_crit.to_csv(os.path.join(PASTA_EXTRATOS, "sensibilidade_sh4_criterio_resumo.csv"), index=False)
    painel_crit.to_csv(os.path.join(PASTA_EXTRATOS, "sensibilidade_sh4_criterio_series.csv"))
    print("\nSalvo em extratos: diagnostico_8906.csv, series_sh4_criterio_base_principal.csv,")
    print("sensibilidade_sh4_criterio_resumo.csv e sensibilidade_sh4_criterio_series.csv")


## Observações

- A série em quilograma soma mercadorias heterogêneas e serve só como contraprova do efeito preço (seção 4.4.4 da monografia).
- A sensibilidade à massa mínima do vetor (`MASSA_MIN_KG`) não foi testada.
- `KG_LIQUIDO` pode conter erros de registro, como o da posição 8906 em 2022. A exclusão do capítulo 89 resolve esse caso, e nenhuma posição passa de 16,5% da série de produto em nenhum ano-base.